# Paper Simulation Figures

This notebook collects paper-ready figures comparing experimental traces with posterior predictive simulations.

In [ ]:
GENERATE_PANEL_A = False # False # True
GENERATE_PANEL_B_POSTERIOR_RIDGES = False # False # True # Posteriors over parameters
GENERATE_PANEL_C_SIM_PIPELINE_EXAMPLE = False
GENERATE_PANEL_FEATURE_MATCH_CORRELATION = False # accuracy of posterior predictive checks
GENERATE_PANEL_FEATURE_PARAMETER_CORRELATION = True # Param-features correlations across simulations

# Central figure-size controls. Set any value to a (width, height) tuple in inches, or None to use that panel's default.
FIGURE_SIZE_OVERRIDES = {
    "panel_a": (7.2, 2.6), # (21, 7), # (7.2, 2.6), # None,
    "panel_b_posterior_ridges": (3.3, 2.1), # None,
    "panel_c_input": (1.9, 2.1), # (10, 8), #(1.9, 2.1), # None,
    "panel_c_spike_cst": (1.9, 2.1), # (10, 8), #(1.9, 1.0), # None,
    "panel_c_processed": (1.9, 2.1), # (10, 8), #(1.9, 2.1), # None,
    "feature_match_correlation": (7.2, 2.0), # None,
    "feature_parameter_correlation": None,
}

# Central subplot geometry controls. Set entries to tuples/floats, or leave as None/default values.
SUBPLOT_SIZE_CONTROLS = {
    "panel_a_curve_width_ratios": None,
    "panel_c_spike_cst_height_ratios": None,
    "feature_match_correlation_width_ratios": None,
}

# Shared visual defaults used by panel-specific style dictionaries unless they explicitly override them.
GLOBAL_FIGURE_DPI = 600
GLOBAL_SAVE_BBOX = None # "tight"
GLOBAL_SAVE_PAD_INCHES = 0.0 # 0.005 # 0.02

# Keep text editable in PDF/SVG outputs for Illustrator editing.
# PDF/PS fonttype 42 embeds TrueType fonts instead of converting text to Type 3/path glyphs.
EMBED_EDITABLE_TEXT_IN_VECTOR_OUTPUTS = True
VECTOR_TEXT_FONT_SETTINGS = {
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
GLOBAL_TITLE_FONTSIZE = 2.0 # 10.0
GLOBAL_AXIS_LABEL_FONTSIZE = 3.0 # 8.0
GLOBAL_TICK_LABELSIZE = 2.5 # 7.0
GLOBAL_LEGEND_FONTSIZE = 3.0 # 7.0
GLOBAL_ANNOTATION_FONTSIZE = 3.0 # 7.0
GLOBAL_GRID_ALPHA = 0.22
GLOBAL_TIGHT_LAYOUT = False # True
GLOBAL_SUBPLOTS_ADJUST = {"left": None, "right": None, "bottom": None, "top": None, "wspace": None, "hspace": None}


def _figure_size(key, default):
    value = FIGURE_SIZE_OVERRIDES.get(str(key), None) if isinstance(FIGURE_SIZE_OVERRIDES, dict) else None
    return default if value is None else tuple(value)


def _subplot_control(key, default):
    value = SUBPLOT_SIZE_CONTROLS.get(str(key), None) if isinstance(SUBPLOT_SIZE_CONTROLS, dict) else None
    return default if value is None else value


In [ ]:
from pathlib import Path
import pickle
from statistics import NormalDist
import warnings

import h5py

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgba
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

from experimental_across_condition_summary import (
    ROW_SPECS,
    _auto_limits,
    _build_subject_records,
    _draw_time_markers,
    _load_export_table,
)
from posterior_predictive_across_condition_summary import (
    _coerce_condition_batch_roots,
    _build_condition_records,
)
from parameter_set_batch_summary import _interp_to_reference, load_parameter_set_run
from sbi_sim_exp_inference import format_feature_label, format_parameter_label

plt.rcParams['figure.facecolor'] = 'white'
if globals().get("EMBED_EDITABLE_TEXT_IN_VECTOR_OUTPUTS", True):
    plt.rcParams.update(globals().get("VECTOR_TEXT_FONT_SETTINGS", {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"}))

from parameter_set_batch_summary import (
    _resolve_feature_config,
    fit_baseline_spline_and_extract_feature,
)


# General visual config (all plots)

In [ ]:
# -----------------------------
# EDITABLE CONFIGURATION
# -----------------------------
CONDITION_ORDER = ["TA_NOGO", "HAND_NOGO", "HAND_GO"]

EXPERIMENTAL_ANALYZED_ROOT = Path("EXP_DATA_ANALYZED_HDF5")
EXPERIMENTAL_EXPORT_TABLE_PATH = EXPERIMENTAL_ANALYZED_ROOT / "_batch_summaries" / "experimental_batch_feature_summary.pkl"

CONDITION_BATCH_ROOTS = {
    "HAND_GO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_GO_sbi_with_all_features"),
    "HAND_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_NOGO_sbi_with_all_features"),
    "TA_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_TA_NOGO_sbi_with_all_features"),
}
POSTERIOR_PREDICTIVE_SUMMARY_SUBDIR_NAME = "parameter_set_summaries"

INCLUDE_FORCE_ROW = True
FORCE_EXPERIMENTAL_DATA_PATH = EXPERIMENTAL_ANALYZED_ROOT / "TORQUE_experimental_data.csv"
FORCE_TRACE_KEY = "force"
FORCE_TIME_COL = "time_sec"
FORCE_VALUE_COL = "force_mean"
FORCE_ID_COL = "id"
FORCE_CONDITION_COL = "condition"
FORCE_N_TRIALS_COL = "n_trials"
# Force burst/trend feature extraction. These values intentionally mirror the CST/sync feature extraction windows.
# Units of exported force features are % MVC because force_mean is multiplied by FORCE_SCALE_FACTOR below.
FORCE_FEATURE_CONFIG = {
    "enabled": True,
    "fitting_window": (-3.0, 1.0),
    "analysis_window": (0.2, 0.7), # Different to take electromechanical delay into account # (0.0, 0.5),
    "buffer_s": 0.05,
    "spline_smoothing": None,
    "spline_order": 4,
    "n_knots": None,
    "trend_pre_window": (-3.0, -1.0),
    "trend_post_window": (0.0, 1.0),
    "extrapolate": False,
}
FORCE_SCALE_FACTOR = 100.0  # CSV force_mean is fractional MVC; plot/export as % MVC.
FORCE_FEATURE_UNIT = "% MVC"

FEATURE_ROW_ORDER = ["force", "cst_lf", "cst_alpha_mod", "cst_beta_mod", "sync_trace"]
# Backward-compatible aliases for earlier notebook cells/config snapshots.
INCLUDE_TORQUE_ROW = INCLUDE_FORCE_ROW
TORQUE_EXPERIMENTAL_DATA_PATH = FORCE_EXPERIMENTAL_DATA_PATH
TORQUE_TRACE_KEY = FORCE_TRACE_KEY
TORQUE_TIME_COL = FORCE_TIME_COL
TORQUE_VALUE_COL = FORCE_VALUE_COL
TORQUE_ID_COL = FORCE_ID_COL
TORQUE_CONDITION_COL = FORCE_CONDITION_COL
TORQUE_N_TRIALS_COL = FORCE_N_TRIALS_COL
TORQUE_FEATURE_CONFIG = FORCE_FEATURE_CONFIG

# "posterior_predictive_mean" = numeric posterior samples; "posterior_mode" = posterior-mode parameter-set folder only.
SIM_TRACE_SOURCE = "posterior_predictive_mean"

# Central tendency of the simulation overlay across selected posterior simulation traces.
# Accepted: "mean", "median", "mode". The mode is an empirical per-timepoint histogram mode.
SIM_TRACE_STATISTIC = "mean" # "median" # "mean"
SIM_TRACE_MODE_BINS = 25

# Optional uncertainty display around the simulation central trace.
# Accepted: None, "sd", "iqr"/"p25_p75", or "percentile" using SIM_TRACE_BAND_PERCENTILES.
SIM_TRACE_BAND = None # "percentile" # "percentile"
SIM_TRACE_BAND_PERCENTILES = (25.0, 75.0)

# Central tendency of the experimental participant traces. Accepted: "mean", "median".
EXPERIMENTAL_TRACE_STATISTIC = "mean"
SHOW_EXPERIMENTAL_INDIVIDUAL_TRACES = False

# Optional shaded band across experimental participant traces.
# Accepted: None, "sd", "iqr", "percentile", "ci".
EXPERIMENTAL_TRACE_BAND = "ci" # Accepted: None, "sd", "iqr", "percentile", "ci".
EXPERIMENTAL_TRACE_BAND_PERCENTILES = (25.0, 75.0)
EXPERIMENTAL_TRACE_CI_LEVEL = 0.95

# Mask possible edge artifacts on the displayed experimental summary curve only.
# Participant traces and experimental shaded bands keep the full trace.
EXPERIMENTAL_EDGE_MASK_MS_BY_TRACE = {
    "force": 0.0,
    "cst_lf": 50.0,
    "cst_alpha_mod": 0.0,
    "cst_beta_mod": 0.0,
    "sync_trace": 0.0,
}

SHOW_FEATURE_INSETS = True
# Optional y-limit overrides for the right-side Burst/Trend feature insets.
# Keys are row names, e.g. {"cst_lf": (-1, 4), "sync_trace": (-0.2, 0.8)}.
FEATURE_YLIM_OVERRIDES = {
    "cst_alpha_mod": (-0.6, 1.0),
    "cst_beta_mod": (-0.3, 0.5),
    "sync_trace": (-0.1, 0.25),
}

DISPLAY_XLIM_REL_CUE_S = (-3.0, 1.0)
PARTICIPANT_IDS = None
SESSION_TAGS = None
SELECTED_PARAMETER_SET_IDS = None
PARAMETER_SET_ID_RANGE = None

OUTPUT_DIR = Path("paper_figures")
SAVE_FORCE_FEATURE_TABLE = True
FORCE_FEATURE_TABLE_PATH = OUTPUT_DIR / "panel_A_force_burst_trend_features.csv"
# GENERATE_PANEL_A = True
# GENERATE_PANEL_B_POSTERIOR_RIDGES = True
PANEL_A_FILENAME_STEM = "panel_A_experiment_vs_posterior_predictive"
SAVE_FIGURE_FORMATS = ("png", "pdf")
SAVE_PANEL_A_CACHE = True
PANEL_A_CACHE_PATH = OUTPUT_DIR / f"{PANEL_A_FILENAME_STEM}_trace_cache.pkl"

EXPECTED_NUMERIC_PARAMETER_SETS_PER_CONDITION = 51
EXPECTED_REPEATS_PER_PARAMETER_SET = 32
EXPECTED_POSTERIOR_MODE_SETS_PER_CONDITION = 1
EXPECTED_POSTERIOR_MODE_REPEATS = 32

ROW_COLORS = {
    "force": "#267346",
    "cst_lf": "#ff3300",
    "cst_alpha_mod": "#ffa600",
    "cst_beta_mod": "#008cff",
    "sync_trace": "#a51e9e",
}

ROW_LABELS = {
    "cst_lf": "Low-frequency CST\n(spikes/s/MU)",
    "cst_alpha_mod": "Alpha modulation\n(z)",
    "cst_beta_mod": "Beta modulation\n(z)",
    "sync_trace": "Rate-controlled\nsynchrony",
    "force": "Force\n(% MVC)",
}

FIGURE_STYLE = {
    "figure_size": _figure_size("panel_a", (17.0, 12.0)),
    "figure_dpi": GLOBAL_FIGURE_DPI,
    "participant_trace_lw": 0.5,
    "participant_trace_alpha": 0.2,
    "participant_trace_linestyle": "-",
    "experimental_mean_lw": 1.0,
    "experimental_mean_linestyle": "-",
    "simulation_mean_lw": 1.0,
    "simulation_mean_linestyle": "-",
    "simulation_mean_color": "black",
    "simulation_mean_alpha": 0.5,
    "simulation_band_color": "black",
    "simulation_band_alpha": 0.12,
    "participant_trace_zorder": 2,
    "experimental_mean_zorder": 4,
    "simulation_band_zorder": 1,
    "simulation_mean_zorder": 5,
    "experimental_band_alpha": 0.14,
    "experimental_band_zorder": 1.5,
    "curve_width_ratios": _subplot_control("panel_a_curve_width_ratios", (3.4, 1.15)), # 0.95)),
    "pair_wspace": 0.08,
    "feature_violin_width": 0.7,
    "feature_violin_alpha": 0.3,
    "feature_violin_edge_lw": 0.0,
    "feature_violin_edge_alpha": 0.0,
    "feature_point_size": 5,
    "feature_point_alpha": 1.0, # 0.5,
    "feature_point_jitter": 0.2,
    "feature_point_edgecolor": "none",
    "feature_point_linewidth": 0.0,
    "feature_point_zorder": 4,
    "feature_mean_lw": 1.0,
    "feature_mean_alpha": 1.0, # 0.95,
    "feature_mean_half_width": 0.35,
    "feature_mean_zorder": 5,
    "feature_sim_lw": 1.0,
    "feature_sim_alpha": 1.0, # 0.9,
    "feature_sim_half_width": 0.30,
    "feature_sim_zorder": 6,
    "feature_zero_line_color": "#777777",
    "feature_zero_line_lw": 0.8,
    "feature_zero_line_ls": "--",
    "feature_zero_line_alpha": 0.55,
    "feature_zero_line_zorder": 1,
    "feature_xlim": (0.4, 2.6),
    "feature_grid_axis": "y",
    "feature_hide_y_ticklabels_nonlast": True,
    "feature_tick_labelsize": 3, # 7,
    "feature_grid_alpha": 0.2,
    "ready_lw": 0.6,
    "ready_alpha": 0.0,
    "go_lw": 1.0,
    "go_alpha": 0.85,
    "event_color": "#000000",
    "time_tick_step_s": 1.0,
    "condition_title_fontsize": 6, # 11,
    "row_label_fontsize": 4, # 9,
    "axis_label_fontsize": 4, # 9,
    "tick_labelsize": 3, # 8,
    "grid_alpha": 0.22,
    "alpha_beta_zero_line_color": "#777777",
    "alpha_beta_zero_line_lw": 0.8,
    "alpha_beta_zero_line_ls": "--",
    "alpha_beta_zero_line_alpha": 0.6,
    "alpha_beta_zero_line_zorder": 1,
    "legend_fontsize": 3.5, #8,
    "wspace": 0.1, # 0.2,
    "hspace": 0.1, # 0.2,
    "condition_wspace": 0.1, # 0.2,
    "main_inset_wspace": 0.1, # 0.2,
    "legend_loc": "upper center",
    "legend_bbox_to_anchor": (0.5, 1.005),
    "legend_ncol_max": 5,
    "legend_frameon": False,
    "subplots_adjust_top": 0.94,
    "xlabel_text": "Time from cue (s)",
}

# None means automatic shared limits by row across experimental and simulation traces.
YLIM_OVERRIDES = {
    "force": (18.5, 20.5),
    "cst_lf": (11, 17),
    "cst_alpha_mod": (-0.5, 2.0),
    "cst_beta_mod": (-0.3, 0.8),
    "sync_trace": (0.1, 0.4),
}


# -----------------------------
# FEATURE-PARAMETER CORRELATION PANEL
# -----------------------------
FEATURE_PARAMETER_CORRELATION_BUNDLE_PATH = Path("$_sbi_output_GONOGO_V3") / "SBI_RUN_normalize_to_exp_FEATURES_all_windows" / "trained_inference_bundle.pkl"
FEATURE_PARAMETER_CORRELATION_FILENAME_STEM = "panel_feature_parameter_correlation_heatmap"
# Lists below control both selection and display order. Set either list to None to use all available entries.
# Parameter order here is top-to-bottom when FEATURE_PARAMETER_CORRELATION_ORIENTATION="parameters_rows".
FEATURE_PARAMETER_CORRELATION_SELECTED_PARAMETERS = [
    "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA",
    "SIM_PARAM_TREND_INPUT_lf_trend_slope_nA_per_s",
    "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA",
    "SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s",
    "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA",
    "SIM_PARAM_TREND_INPUT_beta_trend_slope_nA_per_s",
]
# Feature order here is left-to-right when FEATURE_PARAMETER_CORRELATION_ORIENTATION="parameters_rows".
FEATURE_PARAMETER_CORRELATION_SELECTED_FEATURES = [
    "OBS_BURST_FEATURES_resid_cst_low_frequency",
    "OBS_TREND_FEATURE_cst_low_frequency",
    "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_cst_alpha_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_cst_beta_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_sync_coincidence_curve",
    "OBS_TREND_FEATURE_sync_coincidence_curve",
]
# Accepted: "features_rows" or "parameters_rows".
FEATURE_PARAMETER_CORRELATION_ORIENTATION = "parameters_rows"
FEATURE_PARAMETER_CORRELATION_METHOD = "pearson"
FEATURE_PARAMETER_CORRELATION_STYLE = {
    "figure_size": _figure_size("feature_parameter_correlation", None),
    "figure_dpi": GLOBAL_FIGURE_DPI,
    "cell_size": 0.58,
    # "prinsenvlag" is loaded from cmcrameri when available, with a fallback if the package is missing.
    "cmap": "prinsenvlag",
    "vmin": -1.0,
    "vmax": 1.0,
    "show_cell_values": True,
    "text_fontsize": 7.0,
    "tick_labelsize": 7.0,
    "title_fontsize": 11.0,
    "x_tick_rotation": 45.0,
    "y_tick_rotation": 0.0,
    "square_cells": True,
    "draw_heatmap_as_vector_cells": True,
    "heatmap_edgecolor": "white",
    "heatmap_edge_lw": 0.7,
    "colorbar_shrink": 0.82,
    "colorbar_label": "Pearson r",
    "colorbar_label_fontsize": GLOBAL_AXIS_LABEL_FONTSIZE,
    "colorbar_tick_labelsize": GLOBAL_TICK_LABELSIZE,
    "value_text_abs_threshold": 0.55,
    "tick_length": 3.0,
    "title_text": "Feature-parameter correlations across all SBI training simulations",
    "parameter_colors": {
        "lf": ROW_COLORS.get("cst_lf", "#ff3300"),
        "alpha": ROW_COLORS.get("cst_alpha_mod", "#ffa600"),
        "beta": ROW_COLORS.get("cst_beta_mod", "#9d7aff"),
    },
    "feature_colors": {
        "cst_low_frequency": ROW_COLORS.get("cst_lf", "#ff3300"),
        "cst_alpha": ROW_COLORS.get("cst_alpha_mod", "#ffa600"),
        "cst_beta": ROW_COLORS.get("cst_beta_mod", "#9d7aff"),
        "sync": ROW_COLORS.get("sync_trace", "#a51e9e"),
    },
    "grid_color": "white",
    "grid_lw": 0.7,
    "subplot_left": 0.24,
    "subplot_right": 0.86,
    "subplot_bottom": 0.28,
    "subplot_top": 0.88,
}


# -----------------------------
# FEATURE-MATCH CORRELATION PANEL
# -----------------------------
FEATURE_MATCH_OUTPUT_DIR = Path("$$$_posterior_predictive_checks_results_GO-NOGO_sbi_with_all_features") / "parameter_set_summaries" / "feature_match"
FEATURE_MATCH_CORRELATION_METRICS_PATH = FEATURE_MATCH_OUTPUT_DIR / "feature_match_simulation_direction_metrics.pkl"
FEATURE_MATCH_CORRELATION_FILENAME_STEM = "panel_feature_match_correlation_kde"
FEATURE_MATCH_CORRELATION_STYLE = {
    "figure_size": _figure_size("feature_match_correlation", (10.5, 3.2)),
    "figure_dpi": GLOBAL_FIGURE_DPI,
    "kde_points": 300,
    "xlim": (0.0, 1.0),
    "density_lw": 1.7,
    "density_alpha": 0.95,
    "fill_alpha": 0.18,
    "mean_lw": 1.5,
    "mean_ls": "--",
    "grid_alpha": 0.22,
    "alpha_beta_zero_line_color": "#777777",
    "alpha_beta_zero_line_lw": 0.8,
    "alpha_beta_zero_line_ls": "--",
    "alpha_beta_zero_line_alpha": 0.6,
    "alpha_beta_zero_line_zorder": 1,
    "title_fontsize": 10,
    "axis_label_fontsize": 9,
    "tick_labelsize": 8,
    "legend_fontsize": GLOBAL_LEGEND_FONTSIZE,
    "legend_loc": "upper left",
    "legend_frameon": False,
    "suptitle_text": "Posterior-predictive feature-vector correlation with experimental target",
    "suptitle_fontsize": 11,
    "suptitle_y": None,
    "xlabel_text": "Pearson r",
    "ylabel_text": "Density",
    "tight_layout": True,
    "subplots_adjust": {},
    "width_ratios": _subplot_control("feature_match_correlation_width_ratios", None),
}


# -----------------------------
# PANEL B: posterior marginal ridge plots
# -----------------------------
SBI_POSTERIOR_SAMPLES_PATH = Path("$_sbi_output_GONOGO_V3") / "condition_mean__run_0004" / "posterior_samples.pkl"
PANEL_B_FILENAME_STEM = "panel_B_posterior_marginal_ridges"
PANEL_B_PARAMETER_GRID = [
    ["SIM_PARAM_BURST_INPUT_lf_burst_peak_nA", "SIM_PARAM_TREND_INPUT_lf_trend_slope_nA_per_s"],
    ["SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA", "SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s"],
    ["SIM_PARAM_BURST_INPUT_beta_burst_peak_nA", "SIM_PARAM_TREND_INPUT_beta_trend_slope_nA_per_s"],
]
PANEL_B_ROW_LABELS = ["Low-frequency", "Alpha", "Beta"]
PANEL_B_COLUMN_LABELS = ["Burst peak", "Trend slope"]
PANEL_B_PARAMETER_LABELS = {
    "SIM_PARAM_TREND_INPUT_lf_trend_slope_nA_per_s": "LF trend\n(nA/s)",
    "SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s": "Alpha trend\n(nA/s)",
    "SIM_PARAM_TREND_INPUT_beta_trend_slope_nA_per_s": "Beta trend\n(nA/s)",
    "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA": "LF burst\n(nA)",
    "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA": "Alpha burst\n(nA)",
    "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA": "Beta burst\n(nA)",
}
PANEL_B_CONDITION_COLORS = {
    "TA_NOGO": "#ff3a8d",
    "HAND_NOGO": "#ff7900",
    "HAND_GO": "#1fcb97",
}
PANEL_B_STYLE = {
    "figure_size": _figure_size("panel_b_posterior_ridges", (9.0, 8.0)),
    "figure_dpi": GLOBAL_FIGURE_DPI,
    "kde_points": 300,
    "kde_bandwidth_scale": 1.0,
    "ridge_density_scale": 3.0, # 0.82,
    "ridge_offset_step": 0.5, # 0.42,
    "density_linewidth": 1.0,
    "density_fill_alpha": 0.3,
    "density_line_alpha": 1.0,
    # Ridge draw order: lower ridges are drawn above higher ridges, so overlapping panels read from top to bottom.
    "density_zorder_base": 2.0,
    "density_zorder_step": 1.0,
    "p_gt_zero_text_fontsize": 3.0,
    "p_gt_zero_text_x": 0.98,
    "zero_line_color": "#333333",
    "zero_line_lw": 1.0,
    "zero_line_alpha": 0.75,
    "zero_line_ls": ":",
    "row_label_fontsize": 4, # 10,
    "column_label_fontsize": 4, # 10,
    "axis_label_fontsize": 4, # 8.5,
    "tick_labelsize": 4, # 7.5,
    "legend_fontsize": 3, # 8,
    "legend_loc": "upper center",
    "legend_bbox_to_anchor": (0.5, 1.005),
    "legend_ncol": None,
    "legend_frameon": False,
    "axis_title_loc": "center",
    "tick_length": 2.0,
    "spine_visible": True,
    "suptitle_text": None,
    "suptitle_fontsize": GLOBAL_TITLE_FONTSIZE,
    "suptitle_y": None,
    "grid_alpha": 0.18,
    "wspace": 0.28,
    "hspace": 0.38,
    "subplots_adjust_top": 0.9, # 0.93,
}


## Panel A
Columns are experimental conditions. Rows show low-frequency CST, normalized alpha modulation, normalized beta modulation, and rate-controlled synchrony. Thin colored traces are participant-level experimental averages, thick colored traces are experimental means across participants, and the black trace is the posterior predictive simulation mean.

In [ ]:
def _row_specs_for_panel_a():
    base_by_key = {spec["key"]: dict(spec) for spec in ROW_SPECS}
    if INCLUDE_FORCE_ROW:
        base_by_key[FORCE_TRACE_KEY] = {
            "key": FORCE_TRACE_KEY,
            "time_key": f"{FORCE_TRACE_KEY}_time",
            "title": "Force",
            "ylabel": ROW_LABELS.get(FORCE_TRACE_KEY, "Force\n(% MVC)"),
            "burst_col": "FORCE_BURST_FEATURE_mean_residual",
            "trend_col": "FORCE_TREND_FEATURE_post_minus_pre",
            "has_simulation": False,
        }
    default_order = ["cst_lf", "cst_alpha_mod", "cst_beta_mod", "sync_trace"]
    requested_order = list(FEATURE_ROW_ORDER or default_order)
    specs = []
    for key in requested_order:
        if key == FORCE_TRACE_KEY and not INCLUDE_FORCE_ROW:
            continue
        if key not in base_by_key:
            raise KeyError(f"Unknown Panel A row key in FEATURE_ROW_ORDER: {key!r}")
        spec = dict(base_by_key[key])
        spec["ylabel"] = ROW_LABELS.get(key, spec.get("ylabel", key))
        spec["has_simulation"] = bool(spec.get("has_simulation", key != FORCE_TRACE_KEY))
        specs.append(spec)
    if not specs:
        raise ValueError("FEATURE_ROW_ORDER produced no rows")
    return specs

def _warn_if_unexpected(label, observed, expected):
    if expected is None:
        return
    if int(observed) != int(expected):
        warnings.warn(f"{label}: expected {expected}, observed {observed}")


def _collect_experimental_panel_records():
    export_df = _load_export_table(EXPERIMENTAL_EXPORT_TABLE_PATH)
    records = _build_subject_records(
        EXPERIMENTAL_ANALYZED_ROOT,
        export_df,
        CONDITION_ORDER,
        participant_ids=PARTICIPANT_IDS,
        session_tags=SESSION_TAGS,
    )
    records_by_condition = {condition: [] for condition in CONDITION_ORDER}
    for record in records:
        condition = str(record.get("condition_id"))
        if condition in records_by_condition:
            records_by_condition[condition].append(record)
    for condition in CONDITION_ORDER:
        n_records = len(records_by_condition[condition])
        n_expected = int(export_df[export_df["DATASET_CONTEXT_condition_id"].astype(str) == condition][
            ["DATASET_CONTEXT_participant_id", "DATASET_CONTEXT_session_tag"]
        ].drop_duplicates().shape[0])
        _warn_if_unexpected(f"Experimental participant/session records for {condition}", n_records, n_expected)
        if n_records == 0:
            raise ValueError(f"No experimental participant/session records found for {condition}")
    return records_by_condition, export_df


def _compute_force_features(time_s, force_trace):
    feature_config = _resolve_feature_config(FORCE_FEATURE_CONFIG)
    if not bool(feature_config.get("enabled", True)):
        return {
            "FORCE_BURST_FEATURE_mean_residual": np.nan,
            "FORCE_TREND_FEATURE_post_minus_pre": np.nan,
        }
    try:
        result = fit_baseline_spline_and_extract_feature(
            time_s,
            force_trace,
            fitting_window=feature_config["fitting_window"],
            analysis_window=feature_config["analysis_window"],
            buffer_s=feature_config["buffer_s"],
            spline_smoothing=feature_config["spline_smoothing"],
            spline_order=feature_config["spline_order"],
            n_knots=feature_config["n_knots"],
            trend_pre_window=feature_config.get("trend_pre_window"),
            trend_post_window=feature_config.get("trend_post_window"),
            extrapolate=feature_config["extrapolate"],
            return_debug_info=False,
        )
        return {
            "FORCE_BURST_FEATURE_mean_residual": float(result.feature_mean_residual),
            "FORCE_TREND_FEATURE_post_minus_pre": float(result.trend_post_minus_pre),
        }
    except Exception as exc:
        warnings.warn(f"Could not compute force feature: {exc}")
        return {
            "FORCE_BURST_FEATURE_mean_residual": np.nan,
            "FORCE_TREND_FEATURE_post_minus_pre": np.nan,
        }


def _participant_id_to_force_id(participant_id):
    text = str(participant_id)
    digits = "".join(ch for ch in text if ch.isdigit())
    if digits:
        return f"sub-P{int(digits):03d}"
    return text


def _load_force_records():
    if not INCLUDE_FORCE_ROW:
        return {condition: [] for condition in CONDITION_ORDER}
    path = Path(FORCE_EXPERIMENTAL_DATA_PATH)
    if not path.exists():
        warnings.warn(f"Force row requested, but file was not found: {path}")
        return {condition: [] for condition in CONDITION_ORDER}
    df = pd.read_csv(path)
    required = [FORCE_ID_COL, FORCE_CONDITION_COL, FORCE_TIME_COL, FORCE_VALUE_COL, FORCE_N_TRIALS_COL]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Force CSV is missing required columns: {missing}")
    records_by_condition = {condition: [] for condition in CONDITION_ORDER}
    for (condition, participant_id), group in df.groupby([FORCE_CONDITION_COL, FORCE_ID_COL], sort=False):
        condition = str(condition)
        if condition not in records_by_condition:
            continue
        group = group.sort_values(FORCE_TIME_COL)
        time_s = group[FORCE_TIME_COL].to_numpy(dtype=float)
        force = FORCE_SCALE_FACTOR * pd.to_numeric(group[FORCE_VALUE_COL], errors="coerce").to_numpy(dtype=float)
        finite_time = np.isfinite(time_s)
        if np.count_nonzero(finite_time) < 2:
            continue
        features = _compute_force_features(time_s, force)
        records_by_condition[condition].append({
            "condition_id": condition,
            "participant_id": str(participant_id),
            "session_tag": "force_csv",
            "n_trials": int(pd.to_numeric(group[FORCE_N_TRIALS_COL], errors="coerce").dropna().iloc[0]) if pd.to_numeric(group[FORCE_N_TRIALS_COL], errors="coerce").notna().any() else np.nan,
            "event_times": {"get_ready_cue_s": -3.0, "go_nogo_cue_s": 0.0},
            f"{FORCE_TRACE_KEY}_time": time_s,
            FORCE_TRACE_KEY: force,
            "features": features,
        })
    return records_by_condition


def _force_feature_config_metadata():
    feature_config = _resolve_feature_config(FORCE_FEATURE_CONFIG)

    def _window_or_nan(value):
        if value is None:
            return (np.nan, np.nan)
        try:
            if len(value) == 2:
                return (float(value[0]), float(value[1]))
        except Exception:
            pass
        return (np.nan, np.nan)

    def _scalar_or_nan(value):
        return np.nan if value is None else value

    fitting_window = _window_or_nan(feature_config.get("fitting_window"))
    analysis_window = _window_or_nan(feature_config.get("analysis_window"))
    trend_pre_window = _window_or_nan(feature_config.get("trend_pre_window"))
    trend_post_window = _window_or_nan(feature_config.get("trend_post_window"))
    return {
        "feature_fitting_window": fitting_window,
        "feature_analysis_window": analysis_window,
        "feature_buffer_s": feature_config.get("buffer_s", np.nan),
        "feature_spline_smoothing": _scalar_or_nan(feature_config.get("spline_smoothing")),
        "feature_spline_order": _scalar_or_nan(feature_config.get("spline_order")),
        "feature_n_knots": _scalar_or_nan(feature_config.get("n_knots")),
        "feature_trend_pre_window": trend_pre_window,
        "feature_trend_post_window": trend_post_window,
        "ANALYSIS_PARAM_FEATURE_fitting_window": fitting_window,
        "ANALYSIS_PARAM_FEATURE_analysis_window": analysis_window,
        "ANALYSIS_PARAM_FEATURE_buffer_s": feature_config.get("buffer_s", np.nan),
        "ANALYSIS_PARAM_FEATURE_spline_smoothing": _scalar_or_nan(feature_config.get("spline_smoothing")),
        "ANALYSIS_PARAM_FEATURE_spline_order": _scalar_or_nan(feature_config.get("spline_order")),
        "ANALYSIS_PARAM_FEATURE_n_knots": _scalar_or_nan(feature_config.get("n_knots")),
        "ANALYSIS_PARAM_FEATURE_trend_pre_window": trend_pre_window,
        "ANALYSIS_PARAM_FEATURE_trend_post_window": trend_post_window,
    }


def build_force_feature_table(force_records_by_condition):
    metadata = _force_feature_config_metadata()
    rows = []
    for condition in CONDITION_ORDER:
        for record in (force_records_by_condition or {}).get(condition, []):
            features = dict(record.get("features", {}) or {})
            row = {
                "condition_id": str(record.get("condition_id", condition)),
                "participant_id": str(record.get("participant_id", "")),
                "session_tag": str(record.get("session_tag", "")),
                "n_trials": record.get("n_trials", np.nan),
                "force_trace_key": FORCE_TRACE_KEY,
                "force_scale_factor": float(FORCE_SCALE_FACTOR),
                "force_feature_unit": FORCE_FEATURE_UNIT,
                "FORCE_BURST_FEATURE_mean_residual": features.get("FORCE_BURST_FEATURE_mean_residual", np.nan),
                "FORCE_TREND_FEATURE_post_minus_pre": features.get("FORCE_TREND_FEATURE_post_minus_pre", np.nan),
                "FORCE_BURST_FEATURE_mean_residual_unit": FORCE_FEATURE_UNIT,
                "FORCE_TREND_FEATURE_post_minus_pre_unit": FORCE_FEATURE_UNIT,
                "feature_fitting_window": metadata["feature_fitting_window"],
                "feature_analysis_window": metadata["feature_analysis_window"],
                "feature_buffer_s": metadata["feature_buffer_s"],
                "feature_spline_smoothing": metadata["feature_spline_smoothing"],
                "feature_spline_order": metadata["feature_spline_order"],
                "feature_n_knots": metadata["feature_n_knots"],
                "feature_trend_pre_window": metadata["feature_trend_pre_window"],
                "feature_trend_post_window": metadata["feature_trend_post_window"],
                "ANALYSIS_PARAM_FEATURE_fitting_window": metadata["ANALYSIS_PARAM_FEATURE_fitting_window"],
                "ANALYSIS_PARAM_FEATURE_analysis_window": metadata["ANALYSIS_PARAM_FEATURE_analysis_window"],
                "ANALYSIS_PARAM_FEATURE_buffer_s": metadata["ANALYSIS_PARAM_FEATURE_buffer_s"],
                "ANALYSIS_PARAM_FEATURE_spline_smoothing": metadata["ANALYSIS_PARAM_FEATURE_spline_smoothing"],
                "ANALYSIS_PARAM_FEATURE_spline_order": metadata["ANALYSIS_PARAM_FEATURE_spline_order"],
                "ANALYSIS_PARAM_FEATURE_n_knots": metadata["ANALYSIS_PARAM_FEATURE_n_knots"],
                "ANALYSIS_PARAM_FEATURE_trend_pre_window": metadata["ANALYSIS_PARAM_FEATURE_trend_pre_window"],
                "ANALYSIS_PARAM_FEATURE_trend_post_window": metadata["ANALYSIS_PARAM_FEATURE_trend_post_window"],
            }
            rows.append(row)
    return pd.DataFrame(rows)


def _warn_force_count_mismatches(exp_records_by_condition, force_records_by_condition):
    if not INCLUDE_FORCE_ROW:
        return
    for condition in CONDITION_ORDER:
        exp_ids = {_participant_id_to_force_id(record.get("participant_id")) for record in exp_records_by_condition.get(condition, [])}
        force_ids = {str(record.get("participant_id")) for record in force_records_by_condition.get(condition, [])}
        if exp_ids and force_ids and exp_ids != force_ids:
            warnings.warn(
                f"Force participant IDs differ from HDF5 records for {condition}: "
                f"only_in_hdf5={sorted(exp_ids - force_ids)}, only_in_force={sorted(force_ids - exp_ids)}"
            )


def _build_sim_records_by_condition():
    condition_roots, condition_order = _coerce_condition_batch_roots(CONDITION_BATCH_ROOTS, condition_order=CONDITION_ORDER)
    records_by_condition = {}
    summaries = []
    for condition in condition_order:
        records, summary_df, table_path = _build_condition_records(
            condition,
            condition_roots[condition],
            POSTERIOR_PREDICTIVE_SUMMARY_SUBDIR_NAME,
            selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
            parameter_set_id_range=PARAMETER_SET_ID_RANGE,
        )
        numeric_records = [r for r in records if not r.get("is_posterior_mode", False)]
        mode_records = [r for r in records if r.get("is_posterior_mode", False)]
        _warn_if_unexpected(f"Numeric posterior parameter sets for {condition}", len(numeric_records), EXPECTED_NUMERIC_PARAMETER_SETS_PER_CONDITION)
        _warn_if_unexpected(f"Posterior-mode parameter sets for {condition}", len(mode_records), EXPECTED_POSTERIOR_MODE_SETS_PER_CONDITION)
        for record in numeric_records:
            _warn_if_unexpected(
                f"Repeats in {condition} / parameter_set_{record.get('parameter_set_id')}",
                record.get("n_repeats", 0),
                EXPECTED_REPEATS_PER_PARAMETER_SET,
            )
        for record in mode_records:
            _warn_if_unexpected(
                f"Repeats in {condition} / posterior mode {record.get('mode_observation_label')}",
                record.get("n_repeats", 0),
                EXPECTED_POSTERIOR_MODE_REPEATS,
            )
        records_by_condition[condition] = records
        summaries.append({
            "condition": condition,
            "source_table_path": str(table_path),
            "n_numeric_parameter_sets": len(numeric_records),
            "n_posterior_mode_sets": len(mode_records),
            "n_total_records": len(records),
        })
    return records_by_condition, pd.DataFrame(summaries)


def _selected_simulation_records(records, source):
    source = str(source).strip().lower()
    if source == "posterior_predictive_mean":
        selected = [r for r in records if not r.get("is_posterior_mode", False)]
    elif source == "posterior_mode":
        selected = [r for r in records if r.get("is_posterior_mode", False)]
    else:
        raise ValueError("SIM_TRACE_SOURCE must be 'posterior_predictive_mean' or 'posterior_mode'")
    if not selected:
        raise ValueError(f"No simulation records available for source={source!r}")
    return selected


def _apply_experimental_edge_mask(time, trace, trace_key):
    """Mask only a displayed summary trace; raw participant stack and bands stay intact."""
    time = np.asarray(time, dtype=float)
    trace = np.asarray(trace, dtype=float).copy()
    mask_ms = float(EXPERIMENTAL_EDGE_MASK_MS_BY_TRACE.get(trace_key, 0.0) or 0.0)
    if mask_ms <= 0 or time.size == 0:
        return trace
    finite_time = time[np.isfinite(time)]
    if finite_time.size == 0:
        return trace
    edge_s = mask_ms / 1000.0
    start = float(np.nanmin(finite_time))
    end = float(np.nanmax(finite_time))
    edge_mask = (time <= start + edge_s) | (time >= end - edge_s)
    trace[edge_mask] = np.nan
    return trace


def _stack_experimental_traces(records, trace_key):
    display_start, display_end = map(float, DISPLAY_XLIM_REL_CUE_S)
    ref_time = None
    traces = []
    for record in records:
        time = record.get(f"{trace_key}_time")
        trace = record.get(trace_key)
        if time is None or trace is None:
            continue
        time = np.asarray(time, dtype=float)
        trace = np.asarray(trace, dtype=float)
        mask = (time >= display_start) & (time <= display_end) & np.isfinite(time)
        if np.count_nonzero(mask) < 2:
            continue
        time_zoom = time[mask]
        trace_zoom = trace[mask]
        if ref_time is None:
            ref_time = time_zoom
            traces.append(trace_zoom)
        else:
            traces.append(_interp_to_reference(time_zoom, trace_zoom, ref_time))
    if ref_time is None or not traces:
        raise ValueError(f"No experimental traces available for {trace_key!r}")
    return ref_time, np.vstack(traces)


def _stack_simulation_traces(records, trace_key, source):
    selected = _selected_simulation_records(records, source)
    ref_time = None
    traces = []
    for record in selected:
        time = record.get(f"{trace_key}_time")
        trace = record.get(trace_key)
        if time is None or trace is None:
            continue
        time = np.asarray(time, dtype=float)
        trace = np.asarray(trace, dtype=float)
        finite = np.isfinite(time) & np.isfinite(trace)
        if np.count_nonzero(finite) < 2:
            continue
        time = time[finite]
        trace = trace[finite]
        if ref_time is None:
            ref_time = time
            traces.append(trace)
        else:
            traces.append(_interp_to_reference(time, trace, ref_time))
    if ref_time is None or not traces:
        raise ValueError(f"No finite simulation trace for {trace_key!r}, source={source!r}")
    return ref_time, np.vstack(traces), len(traces)


def _empirical_mode_trace(stack, n_bins=25):
    stack = np.asarray(stack, dtype=float)
    out = np.full(stack.shape[1], np.nan, dtype=float)
    n_bins = max(3, int(n_bins))
    for idx in range(stack.shape[1]):
        values = stack[:, idx]
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        if values.size == 1 or np.nanmin(values) == np.nanmax(values):
            out[idx] = float(values[0])
            continue
        counts, edges = np.histogram(values, bins=n_bins)
        if counts.size == 0 or np.max(counts) == 0:
            out[idx] = float(np.nanmedian(values))
            continue
        best = int(np.argmax(counts))
        out[idx] = 0.5 * (edges[best] + edges[best + 1])
    return out


def _nanmean_axis0_quiet(stack):
    stack = np.asarray(stack, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(stack, axis=0)


def _nanmedian_axis0_quiet(stack):
    stack = np.asarray(stack, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmedian(stack, axis=0)


def _nanstd_axis0_quiet(stack, ddof=0):
    stack = np.asarray(stack, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanstd(stack, axis=0, ddof=ddof)


def _summary_scalar(values, statistic, *, mode_bins=25):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    statistic = str(statistic).strip().lower()
    if statistic == "mean":
        return float(np.nanmean(values))
    if statistic == "median":
        return float(np.nanmedian(values))
    if statistic == "mode":
        if values.size == 1 or np.nanmin(values) == np.nanmax(values):
            return float(values[0])
        counts, edges = np.histogram(values, bins=max(3, int(mode_bins)))
        if counts.size == 0 or np.max(counts) == 0:
            return float(np.nanmedian(values))
        best = int(np.argmax(counts))
        return float(0.5 * (edges[best] + edges[best + 1]))
    raise ValueError("Statistic must be 'mean', 'median', or 'mode'")


def _experimental_center_trace(stack):
    statistic = str(EXPERIMENTAL_TRACE_STATISTIC).strip().lower()
    if statistic == "mean":
        return _nanmean_axis0_quiet(stack)
    if statistic == "median":
        return _nanmedian_axis0_quiet(stack)
    raise ValueError("EXPERIMENTAL_TRACE_STATISTIC must be 'mean' or 'median'")


def _experimental_trace_band(stack):
    band = EXPERIMENTAL_TRACE_BAND
    if band is None or str(band).strip().lower() in ("", "none", "false"):
        return None, None, "none"
    band = str(band).strip().lower()
    center = _experimental_center_trace(stack)
    if band == "sd":
        spread = _nanstd_axis0_quiet(stack, ddof=1 if stack.shape[0] > 1 else 0)
        return center - spread, center + spread, "sd"
    if band == "iqr":
        lower, upper = np.nanpercentile(stack, [25.0, 75.0], axis=0)
        return lower, upper, "iqr"
    if band == "percentile":
        lo, hi = map(float, EXPERIMENTAL_TRACE_BAND_PERCENTILES)
        lower, upper = np.nanpercentile(stack, [lo, hi], axis=0)
        return lower, upper, f"p{lo:g}_p{hi:g}"
    if band == "ci":
        if str(EXPERIMENTAL_TRACE_STATISTIC).strip().lower() != "mean":
            warnings.warn("Experimental CI bands are mean-centered; EXPERIMENTAL_TRACE_STATISTIC is not 'mean'.")
        n = np.sum(np.isfinite(stack), axis=0)
        mean = _nanmean_axis0_quiet(stack)
        sd = _nanstd_axis0_quiet(stack, ddof=1)
        sem = np.divide(sd, np.sqrt(n), out=np.full_like(sd, np.nan, dtype=float), where=n > 1)
        alpha = 1.0 - float(EXPERIMENTAL_TRACE_CI_LEVEL)
        z = NormalDist().inv_cdf(1.0 - 0.5 * alpha)
        return mean - z * sem, mean + z * sem, f"ci{100 * float(EXPERIMENTAL_TRACE_CI_LEVEL):g}"
    raise ValueError("EXPERIMENTAL_TRACE_BAND must be None, 'sd', 'iqr', 'percentile', or 'ci'")


def _simulation_center_and_band(stack):
    statistic = str(SIM_TRACE_STATISTIC).strip().lower()
    if statistic == "mean":
        center = _nanmean_axis0_quiet(stack)
    elif statistic == "median":
        center = _nanmedian_axis0_quiet(stack)
    elif statistic == "mode":
        center = _empirical_mode_trace(stack, SIM_TRACE_MODE_BINS)
    else:
        raise ValueError("SIM_TRACE_STATISTIC must be 'mean', 'median', or 'mode'")

    band = SIM_TRACE_BAND
    if band is None or str(band).strip().lower() in ("", "none", "false"):
        return center, None, None, "none"
    band = str(band).strip().lower()
    if band == "sd":
        spread = _nanstd_axis0_quiet(stack, ddof=1 if stack.shape[0] > 1 else 0)
        return center, center - spread, center + spread, "sd"
    if band in ("iqr", "p25_p75", "25_75"):
        lower, upper = np.nanpercentile(stack, [25.0, 75.0], axis=0)
        return center, lower, upper, "iqr"
    if band == "percentile":
        lo, hi = map(float, SIM_TRACE_BAND_PERCENTILES)
        lower, upper = np.nanpercentile(stack, [lo, hi], axis=0)
        return center, lower, upper, f"p{lo:g}_p{hi:g}"
    raise ValueError("SIM_TRACE_BAND must be None, 'sd', 'iqr', 'p25_p75', or 'percentile'")


def _experimental_trace_summary(records, trace_key):
    time, stack = _stack_experimental_traces(records, trace_key)
    center = _experimental_center_trace(stack)
    center = _apply_experimental_edge_mask(time, center, trace_key)
    lower, upper, band_label = _experimental_trace_band(stack)
    return {"time": time, "stack": stack, "center": center, "band_lower": lower, "band_upper": upper, "band_label": band_label, "n_traces": int(stack.shape[0])}


def _simulation_trace_summary(records, trace_key, source):
    time, stack, n_traces = _stack_simulation_traces(records, trace_key, source)
    center, lower, upper, band_label = _simulation_center_and_band(stack)
    return {
        "time": time,
        "stack": stack,
        "center": center,
        "band_lower": lower,
        "band_upper": upper,
        "band_label": band_label,
        "n_traces": int(n_traces),
        "statistic": str(SIM_TRACE_STATISTIC).strip().lower(),
    }


def _feature_values(records, spec):
    burst = np.asarray([record.get("features", {}).get(spec["burst_col"], np.nan) for record in records], dtype=float)
    trend = np.asarray([record.get("features", {}).get(spec["trend_col"], np.nan) for record in records], dtype=float)
    return burst[np.isfinite(burst)], trend[np.isfinite(trend)]


def _feature_ylim_for_spec(exp_records_by_condition, sim_records_by_condition, spec, force_records_by_condition=None):
    key = spec["key"]
    if key in FEATURE_YLIM_OVERRIDES and FEATURE_YLIM_OVERRIDES[key] is not None:
        return tuple(map(float, FEATURE_YLIM_OVERRIDES[key]))
    values = []
    for condition in CONDITION_ORDER:
        records_for_key = (force_records_by_condition or {}).get(condition, []) if spec["key"] == FORCE_TRACE_KEY else exp_records_by_condition.get(condition, [])
        exp_burst, exp_trend = _feature_values(records_for_key, spec)
        values.extend([exp_burst, exp_trend])
        if bool(spec.get("has_simulation", True)):
            try:
                sim_records = _selected_simulation_records(sim_records_by_condition.get(condition, []), SIM_TRACE_SOURCE)
                sim_burst, sim_trend = _feature_values(sim_records, spec)
                values.extend([sim_burst, sim_trend])
            except Exception as exc:
                warnings.warn(f"Could not include simulation feature values in inset limits for {condition}/{key}: {exc}")
    finite_parts = [np.asarray(v, dtype=float)[np.isfinite(v)] for v in values if np.asarray(v).size > 0]
    finite_parts = [v for v in finite_parts if v.size > 0]
    finite = np.concatenate(finite_parts) if finite_parts else np.asarray([])
    return _auto_limits(finite if finite.size else np.asarray([0.0]), zero_floor=False)


def _compute_panel_a_y_limits(exp_records_by_condition, sim_records_by_condition, row_specs, force_records_by_condition=None):
    limits = {"trace": {}, "feature": {}}
    for spec in row_specs:
        key = spec["key"]
        if key in YLIM_OVERRIDES and YLIM_OVERRIDES[key] is not None:
            limits["trace"][key] = tuple(map(float, YLIM_OVERRIDES[key]))
        else:
            values = []
            for condition in CONDITION_ORDER:
                try:
                    records_for_key = (force_records_by_condition or {}).get(condition, []) if key == FORCE_TRACE_KEY else exp_records_by_condition.get(condition, [])
                    exp_summary = _experimental_trace_summary(records_for_key, key)
                    values.append(exp_summary["center"])
                    if exp_summary["band_lower"] is not None:
                        values.extend([exp_summary["band_lower"], exp_summary["band_upper"]])
                    if bool(spec.get("has_simulation", True)):
                        sim_summary = _simulation_trace_summary(sim_records_by_condition.get(condition, []), key, SIM_TRACE_SOURCE)
                        values.append(sim_summary["center"])
                        if sim_summary["band_lower"] is not None:
                            values.extend([sim_summary["band_lower"], sim_summary["band_upper"]])
                except Exception as exc:
                    warnings.warn(f"Could not include traces in y-limit calculation for {condition}/{key}: {exc}")
            finite = np.concatenate([np.asarray(v, dtype=float).ravel() for v in values if np.asarray(v).size > 0]) if values else np.asarray([])
            finite = finite[np.isfinite(finite)]
            limits["trace"][key] = _auto_limits(finite if finite.size else np.asarray([0.0]), zero_floor=False)
        limits["feature"][key] = _feature_ylim_for_spec(exp_records_by_condition, sim_records_by_condition, spec, force_records_by_condition=force_records_by_condition)
    return limits


def _plot_feature_inset(ax, exp_records, sim_records, spec, color, y_limits):
    positions = [1.0, 2.0]
    labels = ["Burst", "Trend"]
    exp_datasets = list(_feature_values(exp_records, spec))
    valid_positions = [pos for pos, vals in zip(positions, exp_datasets) if vals.size > 0]
    valid_data = [vals for vals in exp_datasets if vals.size > 0]
    if valid_data:
        violin = ax.violinplot(valid_data, positions=valid_positions, widths=FIGURE_STYLE["feature_violin_width"], showmeans=False, showextrema=False, showmedians=False)
        violin_edge_lw = float(FIGURE_STYLE.get("feature_violin_edge_lw", 0.0))
        for body in violin["bodies"]:
            body.set_facecolor(color)
            if violin_edge_lw > 0:
                body.set_edgecolor(to_rgba(color, FIGURE_STYLE.get("feature_violin_edge_alpha", 1.0)))
                body.set_linewidth(violin_edge_lw)
            else:
                body.set_edgecolor("none")
                body.set_linewidth(0.0)
            body.set_alpha(FIGURE_STYLE["feature_violin_alpha"])
    rng = np.random.default_rng(0)
    exp_counts = []
    sim_counts = []
    for pos, values in zip(positions, exp_datasets):
        exp_counts.append(int(values.size))
        if values.size == 0:
            continue
        jitter = rng.uniform(-FIGURE_STYLE["feature_point_jitter"], FIGURE_STYLE["feature_point_jitter"], size=values.size)
        point_edge_lw = float(FIGURE_STYLE.get("feature_point_linewidth", 0.0))
        point_edgecolor = FIGURE_STYLE.get("feature_point_edgecolor", "none") if point_edge_lw > 0 else "none"
        ax.scatter(
            np.full(values.shape, pos, dtype=float) + jitter,
            values,
            s=FIGURE_STYLE["feature_point_size"],
            alpha=FIGURE_STYLE["feature_point_alpha"],
            color=color,
            edgecolors=point_edgecolor,
            linewidths=point_edge_lw,
            zorder=FIGURE_STYLE["feature_point_zorder"],
        )
        mean_value = float(np.nanmean(values))
        half_width = float(FIGURE_STYLE["feature_mean_half_width"])
        ax.hlines(mean_value, pos - half_width, pos + half_width, color=color, lw=FIGURE_STYLE["feature_mean_lw"], alpha=FIGURE_STYLE["feature_mean_alpha"], zorder=FIGURE_STYLE["feature_mean_zorder"])
    if bool(spec.get("has_simulation", True)):
        try:
            selected_sim_records = _selected_simulation_records(sim_records, SIM_TRACE_SOURCE)
            sim_datasets = list(_feature_values(selected_sim_records, spec))
        except Exception:
            sim_datasets = [np.asarray([], dtype=float), np.asarray([], dtype=float)]
    else:
        sim_datasets = [np.asarray([], dtype=float), np.asarray([], dtype=float)]
    for pos, values in zip(positions, sim_datasets):
        sim_counts.append(int(values.size))
        if values.size == 0:
            continue
        sim_value = _summary_scalar(values, SIM_TRACE_STATISTIC, mode_bins=SIM_TRACE_MODE_BINS)
        if np.isfinite(sim_value):
            half_width = float(FIGURE_STYLE["feature_sim_half_width"])
            ax.hlines(sim_value, pos - half_width, pos + half_width, color=FIGURE_STYLE["simulation_mean_color"], lw=FIGURE_STYLE["feature_sim_lw"], alpha=FIGURE_STYLE["feature_sim_alpha"], zorder=FIGURE_STYLE["feature_sim_zorder"])
    ax.axhline(0.0, color=FIGURE_STYLE["feature_zero_line_color"], lw=FIGURE_STYLE["feature_zero_line_lw"], ls=FIGURE_STYLE["feature_zero_line_ls"], alpha=FIGURE_STYLE["feature_zero_line_alpha"], zorder=FIGURE_STYLE["feature_zero_line_zorder"])
    ax.set_xlim(*FIGURE_STYLE["feature_xlim"])
    ax.set_ylim(*map(float, y_limits))
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=FIGURE_STYLE["feature_tick_labelsize"])
    ax.tick_params(axis="both", labelsize=FIGURE_STYLE["feature_tick_labelsize"])
    ax.grid(alpha=FIGURE_STYLE["feature_grid_alpha"], axis=FIGURE_STYLE["feature_grid_axis"])
    return {"n_exp_burst": exp_counts[0], "n_exp_trend": exp_counts[1], "n_sim_burst": sim_counts[0] if sim_counts else 0, "n_sim_trend": sim_counts[1] if len(sim_counts) > 1 else 0}


def _save_figure(fig, output_dir, stem, formats):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(formats, str):
        formats = (formats,)
    paths = []
    for fmt in formats:
        fmt = str(fmt).lstrip(".").strip()
        if not fmt:
            continue
        path = output_dir / f"{stem}.{fmt}"
        fig.savefig(path, dpi=fig.dpi, bbox_inches=GLOBAL_SAVE_BBOX, pad_inches=GLOBAL_SAVE_PAD_INCHES)
        paths.append(str(path))
    return paths


In [ ]:
def plot_panel_a(exp_records_by_condition, sim_records_by_condition, *, force_records_by_condition=None, output_dir=None):
    row_specs = _row_specs_for_panel_a()
    y_limits = _compute_panel_a_y_limits(
        exp_records_by_condition,
        sim_records_by_condition,
        row_specs,
        force_records_by_condition=force_records_by_condition,
    )
    n_rows = len(row_specs)
    n_cols = len(CONDITION_ORDER)
    fig = plt.figure(figsize=FIGURE_STYLE["figure_size"], dpi=FIGURE_STYLE["figure_dpi"])
    outer = fig.add_gridspec(
        n_rows,
        n_cols,
        wspace=FIGURE_STYLE.get("condition_wspace", FIGURE_STYLE["wspace"]),
        hspace=FIGURE_STYLE["hspace"],
    )
    display_start, display_end = map(float, DISPLAY_XLIM_REL_CUE_S)
    panel_summary_rows = []

    for col_idx, condition in enumerate(CONDITION_ORDER):
        base_exp_records = exp_records_by_condition.get(condition, [])
        sim_records = sim_records_by_condition.get(condition, [])
        if not base_exp_records:
            raise ValueError(f"No experimental records available for condition {condition}")
        for row_idx, spec in enumerate(row_specs):
            key = spec["key"]
            exp_records = (force_records_by_condition or {}).get(condition, []) if key == FORCE_TRACE_KEY else base_exp_records
            if not exp_records:
                raise ValueError(f"No experimental records available for condition {condition}/{key}")
            color = ROW_COLORS.get(key, "#333333")
            if SHOW_FEATURE_INSETS:
                sub = outer[row_idx, col_idx].subgridspec(
                    1,
                    2,
                    width_ratios=FIGURE_STYLE["curve_width_ratios"],
                    wspace=FIGURE_STYLE.get("main_inset_wspace", FIGURE_STYLE["pair_wspace"]),
                )
                ax = fig.add_subplot(sub[0, 0])
                ax_feat = fig.add_subplot(sub[0, 1])
            else:
                ax = fig.add_subplot(outer[row_idx, col_idx])
                ax_feat = None

            exp_summary = _experimental_trace_summary(exp_records, key)
            exp_time = exp_summary["time"]
            if SHOW_EXPERIMENTAL_INDIVIDUAL_TRACES:
                for trace in exp_summary["stack"]:
                    ax.plot(
                        exp_time,
                        trace,
                        color=color,
                        lw=FIGURE_STYLE["participant_trace_lw"],
                        alpha=FIGURE_STYLE["participant_trace_alpha"],
                        linestyle=FIGURE_STYLE["participant_trace_linestyle"],
                        zorder=FIGURE_STYLE["participant_trace_zorder"],
                    )
            if exp_summary["band_lower"] is not None:
                ax.fill_between(
                    exp_time,
                    exp_summary["band_lower"],
                    exp_summary["band_upper"],
                    color=color,
                    alpha=FIGURE_STYLE["experimental_band_alpha"],
                    linewidth=0.0,
                    zorder=FIGURE_STYLE["experimental_band_zorder"],
                )
            ax.plot(
                exp_time,
                exp_summary["center"],
                color=color,
                lw=FIGURE_STYLE["experimental_mean_lw"],
                linestyle=FIGURE_STYLE["experimental_mean_linestyle"],
                alpha=1.0,
                zorder=FIGURE_STYLE["experimental_mean_zorder"],
            )

            if bool(spec.get("has_simulation", True)):
                sim_summary = _simulation_trace_summary(sim_records, key, SIM_TRACE_SOURCE)
                sim_time = sim_summary["time"]
                sim_mask = (sim_time >= display_start) & (sim_time <= display_end)
                if not np.any(sim_mask):
                    raise ValueError(f"No simulation trace samples in display window for {condition}/{key}")
                if sim_summary["band_lower"] is not None:
                    ax.fill_between(
                        sim_time[sim_mask],
                        sim_summary["band_lower"][sim_mask],
                        sim_summary["band_upper"][sim_mask],
                        color=FIGURE_STYLE["simulation_band_color"],
                        alpha=FIGURE_STYLE["simulation_band_alpha"],
                        linewidth=0.0,
                        zorder=FIGURE_STYLE["simulation_band_zorder"],
                    )
                ax.plot(
                    sim_time[sim_mask],
                    sim_summary["center"][sim_mask],
                    color=FIGURE_STYLE["simulation_mean_color"],
                    lw=FIGURE_STYLE["simulation_mean_lw"],
                    linestyle=FIGURE_STYLE["simulation_mean_linestyle"],
                    alpha=FIGURE_STYLE["simulation_mean_alpha"],
                    zorder=FIGURE_STYLE["simulation_mean_zorder"],
                )
                n_sim_traces = int(sim_summary["n_traces"])
                sim_trace_source = SIM_TRACE_SOURCE
                sim_trace_statistic = sim_summary["statistic"]
                sim_trace_band = sim_summary["band_label"]
            else:
                sim_summary = {"n_traces": 0, "statistic": "none", "band_label": "none"}
                n_sim_traces = 0
                sim_trace_source = "none"
                sim_trace_statistic = "none"
                sim_trace_band = "none"

            feature_counts = {"n_exp_burst": np.nan, "n_exp_trend": np.nan, "n_sim_burst": np.nan, "n_sim_trend": np.nan}
            if ax_feat is not None:
                feature_counts = _plot_feature_inset(ax_feat, exp_records, sim_records, spec, color, y_limits["feature"][key])
                if col_idx < n_cols - 1 and bool(FIGURE_STYLE["feature_hide_y_ticklabels_nonlast"]):
                    ax_feat.tick_params(axis="y", labelleft=False)

            if exp_records and exp_records[0].get("event_times") is not None:
                _draw_time_markers(ax, exp_records[0]["event_times"], FIGURE_STYLE)
            if key in ("cst_alpha_mod", "cst_beta_mod"):
                ax.axhline(0.0, color=FIGURE_STYLE["alpha_beta_zero_line_color"], lw=FIGURE_STYLE["alpha_beta_zero_line_lw"], ls=FIGURE_STYLE["alpha_beta_zero_line_ls"], alpha=FIGURE_STYLE["alpha_beta_zero_line_alpha"], zorder=FIGURE_STYLE["alpha_beta_zero_line_zorder"])
            ax.set_xlim(display_start, display_end)
            ax.set_ylim(*y_limits["trace"][key])
            ax.grid(alpha=FIGURE_STYLE["grid_alpha"])
            ax.tick_params(axis="both", labelsize=FIGURE_STYLE["tick_labelsize"])
            if col_idx == 0:
                ax.set_ylabel(spec.get("ylabel", key), fontsize=FIGURE_STYLE["row_label_fontsize"])
            if row_idx == 0:
                ax.set_title(condition.replace("_", " "), fontsize=FIGURE_STYLE["condition_title_fontsize"])
            if row_idx == n_rows - 1:
                ax.set_xlabel(FIGURE_STYLE["xlabel_text"], fontsize=FIGURE_STYLE["axis_label_fontsize"])
            else:
                ax.set_xticklabels([])

            panel_summary_rows.append({
                "condition": condition,
                "trace_key": key,
                "n_experimental_participant_traces": int(exp_summary["n_traces"]),
                "n_simulation_parameter_sets_or_mode_sets": int(n_sim_traces),
                "show_experimental_individual_traces": bool(SHOW_EXPERIMENTAL_INDIVIDUAL_TRACES),
                "show_feature_insets": bool(SHOW_FEATURE_INSETS),
                "sim_trace_source": sim_trace_source,
                "experimental_trace_statistic": str(EXPERIMENTAL_TRACE_STATISTIC).strip().lower(),
                "experimental_trace_band": exp_summary["band_label"],
                "sim_trace_statistic": sim_trace_statistic,
                "sim_trace_band": sim_trace_band,
                **feature_counts,
            })

    sim_label_prefix = "Posterior predictive" if SIM_TRACE_SOURCE == "posterior_predictive_mean" else "Posterior mode"
    sim_stat_label = str(SIM_TRACE_STATISTIC).strip().lower()
    handles = []
    if SHOW_EXPERIMENTAL_INDIVIDUAL_TRACES:
        handles.append(plt.Line2D([0], [0], color="#666666", lw=FIGURE_STYLE["participant_trace_lw"], linestyle=FIGURE_STYLE["participant_trace_linestyle"], alpha=FIGURE_STYLE["participant_trace_alpha"], label="Participant experimental trace"))
    if EXPERIMENTAL_TRACE_BAND is not None and str(EXPERIMENTAL_TRACE_BAND).strip().lower() not in ("", "none", "false"):
        handles.append(plt.Rectangle((0, 0), 1, 1, color="#666666", alpha=FIGURE_STYLE["experimental_band_alpha"], label=f"Experimental band ({EXPERIMENTAL_TRACE_BAND})"))
    handles.append(plt.Line2D([0], [0], color="#666666", lw=FIGURE_STYLE["experimental_mean_lw"], linestyle=FIGURE_STYLE["experimental_mean_linestyle"], label=f"Experimental {str(EXPERIMENTAL_TRACE_STATISTIC).strip().lower()}"))
    if SIM_TRACE_BAND is not None and str(SIM_TRACE_BAND).strip().lower() not in ("", "none", "false"):
        handles.append(plt.Rectangle((0, 0), 1, 1, color=FIGURE_STYLE["simulation_band_color"], alpha=FIGURE_STYLE["simulation_band_alpha"], label=f"{sim_label_prefix} band ({SIM_TRACE_BAND})"))
    handles.append(plt.Line2D([0], [0], color=FIGURE_STYLE["simulation_mean_color"], lw=FIGURE_STYLE["simulation_mean_lw"], linestyle=FIGURE_STYLE["simulation_mean_linestyle"], label=f"{sim_label_prefix} {sim_stat_label}"))
    fig.legend(handles=handles, loc=FIGURE_STYLE["legend_loc"], ncol=min(int(FIGURE_STYLE["legend_ncol_max"]), len(handles)), frameon=FIGURE_STYLE["legend_frameon"], fontsize=FIGURE_STYLE["legend_fontsize"], bbox_to_anchor=FIGURE_STYLE["legend_bbox_to_anchor"])
    fig.subplots_adjust(top=FIGURE_STYLE["subplots_adjust_top"])
    output_paths = []
    if output_dir is not None:
        output_paths = _save_figure(fig, output_dir, PANEL_A_FILENAME_STEM, SAVE_FIGURE_FORMATS)
    return fig, pd.DataFrame(panel_summary_rows), output_paths


In [ ]:
panel_a_summary_df = pd.DataFrame()
panel_a_paths = []
sim_load_summary_df = pd.DataFrame()
exp_records_by_condition = None
experimental_export_df = None
force_records_by_condition = None
force_feature_table_df = pd.DataFrame()
sim_records_by_condition = None

if GENERATE_PANEL_A:
    exp_records_by_condition, experimental_export_df = _collect_experimental_panel_records()
    force_records_by_condition = _load_force_records()
    force_feature_table_df = build_force_feature_table(force_records_by_condition)
    if SAVE_FORCE_FEATURE_TABLE and INCLUDE_FORCE_ROW and not force_feature_table_df.empty:
        FORCE_FEATURE_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
        force_feature_table_df.to_csv(FORCE_FEATURE_TABLE_PATH, index=False)
    _warn_force_count_mismatches(exp_records_by_condition, force_records_by_condition)
    sim_records_by_condition, sim_load_summary_df = _build_sim_records_by_condition()

    fig_panel_a, panel_a_summary_df, panel_a_paths = plot_panel_a(
        exp_records_by_condition,
        sim_records_by_condition,
        force_records_by_condition=force_records_by_condition,
        output_dir=OUTPUT_DIR,
    )

    if SAVE_PANEL_A_CACHE:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(
            {
                "condition_order": CONDITION_ORDER,
                "include_force_row": INCLUDE_FORCE_ROW,
                "feature_row_order": FEATURE_ROW_ORDER,
                "force_scale_factor": FORCE_SCALE_FACTOR,
                "force_data_path": str(FORCE_EXPERIMENTAL_DATA_PATH),
                "sim_trace_source": SIM_TRACE_SOURCE,
                "experimental_trace_statistic": EXPERIMENTAL_TRACE_STATISTIC,
                "show_experimental_individual_traces": SHOW_EXPERIMENTAL_INDIVIDUAL_TRACES,
                "experimental_trace_band": EXPERIMENTAL_TRACE_BAND,
                "experimental_trace_band_percentiles": EXPERIMENTAL_TRACE_BAND_PERCENTILES,
                "experimental_trace_ci_level": EXPERIMENTAL_TRACE_CI_LEVEL,
                "experimental_edge_mask_ms_by_trace": EXPERIMENTAL_EDGE_MASK_MS_BY_TRACE,
                "show_feature_insets": SHOW_FEATURE_INSETS,
                "feature_ylim_overrides": FEATURE_YLIM_OVERRIDES,
                "sim_trace_statistic": SIM_TRACE_STATISTIC,
                "sim_trace_band": SIM_TRACE_BAND,
                "panel_summary": panel_a_summary_df,
                "simulation_load_summary": sim_load_summary_df,
                "figure_paths": panel_a_paths,
            },
            PANEL_A_CACHE_PATH,
        )

    plt.show()
else:
    print("Panel A generation skipped because GENERATE_PANEL_A=False")

panel_a_summary_df


## Panel B - Marginal posterior ridge plots

This panel shows condition-specific marginal posterior densities for trend and burst parameters, with `P(theta > 0)` displayed beside each ridge.


In [ ]:
def _posterior_kde_1d(values, grid, bandwidth_scale=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.full_like(grid, np.nan, dtype=float)
    if values.size == 1 or np.nanmin(values) == np.nanmax(values):
        center = float(values[0])
        width = max(abs(center) * 0.01, 1.0)
        return np.exp(-0.5 * ((grid - center) / width) ** 2) / (width * np.sqrt(2.0 * np.pi))
    kde = gaussian_kde(values)
    kde.set_bandwidth(kde.factor * float(bandwidth_scale))
    return np.asarray(kde(grid), dtype=float)


def _load_panel_b_posterior_samples():
    path = Path(SBI_POSTERIOR_SAMPLES_PATH)
    if not path.exists():
        raise FileNotFoundError(f"Posterior samples file not found: {path}")
    df = pd.read_pickle(path)
    required = {"observation_label"}
    for row in PANEL_B_PARAMETER_GRID:
        required.update(row)
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Posterior samples are missing required columns: {missing}")
    return df


def _panel_b_parameter_label(parameter_name):
    return PANEL_B_PARAMETER_LABELS.get(str(parameter_name), str(parameter_name))


def plot_panel_b_posterior_marginal_ridges(posterior_samples_df, *, output_dir=None):
    labels = [label for label in CONDITION_ORDER if label in set(posterior_samples_df["observation_label"].astype(str))]
    if not labels:
        raise ValueError("No CONDITION_ORDER labels were found in posterior_samples_df")
    style = dict(PANEL_B_STYLE)
    n_rows = len(PANEL_B_PARAMETER_GRID)
    n_cols = len(PANEL_B_PARAMETER_GRID[0])
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=style["figure_size"],
        dpi=style["figure_dpi"],
        squeeze=False,
    )
    samples = {}
    density_records = {}
    shared_density_ymax = 0.0
    x_limits = {}
    for row in PANEL_B_PARAMETER_GRID:
        for param in row:
            all_values = []
            for label in labels:
                vals = pd.to_numeric(
                    posterior_samples_df.loc[posterior_samples_df["observation_label"].astype(str) == str(label), param],
                    errors="coerce",
                ).to_numpy(dtype=float)
                vals = vals[np.isfinite(vals)]
                samples[(label, param)] = vals
                if vals.size:
                    all_values.append(vals)
            if not all_values:
                x_limits[param] = (-1.0, 1.0)
                continue
            finite_all = np.concatenate(all_values)
            lo, hi = float(np.nanmin(finite_all)), float(np.nanmax(finite_all))
            if np.isclose(lo, hi):
                pad = max(abs(lo) * 0.05, 1.0)
            else:
                pad = 0.04 * (hi - lo)
            lo -= pad
            hi += pad
            x_limits[param] = (lo, hi)
            grid = np.linspace(lo, hi, int(style["kde_points"]))
            for label in labels:
                density = _posterior_kde_1d(samples[(label, param)], grid, bandwidth_scale=style["kde_bandwidth_scale"])
                finite_density = density[np.isfinite(density)]
                if finite_density.size:
                    shared_density_ymax = max(shared_density_ymax, float(np.nanmax(finite_density)))
                density_records[(label, param)] = (grid, density)
    denominator = shared_density_ymax if shared_density_ymax > 0 else 1.0
    offset_step = float(style["ridge_offset_step"])
    ridge_scale = float(style["ridge_density_scale"]) * offset_step
    ridge_offsets = np.arange(len(labels), dtype=float) * offset_step
    p_gt_zero_rows = []
    for row_idx, row in enumerate(PANEL_B_PARAMETER_GRID):
        for col_idx, param in enumerate(row):
            ax = axes[row_idx, col_idx]
            lo, hi = x_limits[param]
            if lo <= 0.0 <= hi:
                ax.axvline(
                    0.0,
                    color=style["zero_line_color"],
                    lw=style["zero_line_lw"],
                    ls=style["zero_line_ls"],
                    alpha=style["zero_line_alpha"],
                    zorder=0,
                )
            for label_idx, label in enumerate(labels):
                color = PANEL_B_CONDITION_COLORS.get(label, f"C{label_idx}")
                offset = ridge_offsets[label_idx]
                grid, density = density_records.get((label, param), (np.asarray([]), np.asarray([])))
                vals = samples.get((label, param), np.asarray([]))
                p_gt_zero = float(np.mean(vals > 0.0)) if vals.size else np.nan
                p_gt_zero_rows.append({
                    "parameter_name": param,
                    "observation_label": label,
                    "p_gt_zero": p_gt_zero,
                    "n_samples": int(vals.size),
                })
                ridge_zorder = float(style.get("density_zorder_base", 2.0)) + (len(labels) - 1 - label_idx) * float(style.get("density_zorder_step", 1.0))
                if grid.size:
                    scaled = density / denominator * ridge_scale
                    scaled[~np.isfinite(scaled)] = np.nan
                    ax.fill_between(grid, offset, offset + scaled, color=color, alpha=style["density_fill_alpha"], linewidth=0, zorder=ridge_zorder)
                    ax.plot(grid, offset + scaled, color=color, lw=style["density_linewidth"], alpha=style["density_line_alpha"], label=label, zorder=ridge_zorder + 0.1)
                p_text = f"P>0={p_gt_zero:.2f}" if np.isfinite(p_gt_zero) else "P>0=NA"
                ax.text(
                    float(style["p_gt_zero_text_x"]),
                    offset + 0.06 * offset_step,
                    p_text,
                    transform=ax.get_yaxis_transform(),
                    ha="right",
                    va="bottom",
                    color=color,
                    fontsize=style["p_gt_zero_text_fontsize"],
                    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.62, "pad": 1.0},
                )
            ax.set_xlim(lo, hi)
            ax.set_ylim(-0.25 * offset_step, ridge_offsets[-1] + ridge_scale + 0.35 * offset_step)
            ax.set_yticks(ridge_offsets)
            ax.set_yticklabels(labels, fontsize=style["tick_labelsize"])
            for tick in ax.get_yticklabels():
                tick.set_color(PANEL_B_CONDITION_COLORS.get(tick.get_text(), "black"))
            ax.set_xlabel(_panel_b_parameter_label(param), fontsize=style["axis_label_fontsize"])
            if col_idx == 0:
                ax.set_ylabel(PANEL_B_ROW_LABELS[row_idx], fontsize=style["row_label_fontsize"])
            if row_idx == 0:
                ax.set_title(PANEL_B_COLUMN_LABELS[col_idx], fontsize=style["column_label_fontsize"], loc=style["axis_title_loc"])
            ax.tick_params(axis="both", labelsize=style["tick_labelsize"], length=style["tick_length"])
            for spine in ax.spines.values():
                spine.set_visible(bool(style["spine_visible"]))
            ax.grid(alpha=style["grid_alpha"])
    handles = [plt.Line2D([0], [0], color=PANEL_B_CONDITION_COLORS.get(label, f"C{i}"), lw=style["density_linewidth"], label=label) for i, label in enumerate(labels)]
    ncol = len(handles) if style["legend_ncol"] is None else int(style["legend_ncol"])
    fig.legend(handles=handles, loc=style["legend_loc"], ncol=ncol, frameon=style["legend_frameon"], fontsize=style["legend_fontsize"], bbox_to_anchor=style["legend_bbox_to_anchor"])
    if style["suptitle_text"]:
        suptitle_kwargs = {"fontsize": style["suptitle_fontsize"]}
        if style["suptitle_y"] is not None:
            suptitle_kwargs["y"] = style["suptitle_y"]
        fig.suptitle(style["suptitle_text"], **suptitle_kwargs)
    fig.subplots_adjust(wspace=style["wspace"], hspace=style["hspace"], top=style["subplots_adjust_top"])
    output_paths = []
    if output_dir is not None:
        output_paths = _save_figure(fig, output_dir, PANEL_B_FILENAME_STEM, SAVE_FIGURE_FORMATS)
    return fig, pd.DataFrame(p_gt_zero_rows), output_paths


panel_b_p_gt_zero_df = pd.DataFrame()
panel_b_paths = []
if GENERATE_PANEL_B_POSTERIOR_RIDGES:
    posterior_samples_df = _load_panel_b_posterior_samples()
    fig_panel_b, panel_b_p_gt_zero_df, panel_b_paths = plot_panel_b_posterior_marginal_ridges(
        posterior_samples_df,
        output_dir=OUTPUT_DIR,
    )
    plt.show()
else:
    print("Panel B generation skipped because GENERATE_PANEL_B_POSTERIOR_RIDGES=False")

panel_b_p_gt_zero_df


## Panel C - Simulation pipeline example

This section generates separate figure elements for a single simulated parameter set: input components, spike/CST output, and processed CST/synchrony features with the same spline residual extraction used in the batch summaries.


In [ ]:
# -----------------------------
# PANEL C CONFIGURATION
# -----------------------------
PANEL_C_EXAMPLE_PARAMETER_SET_DIR = Path("paper_figures/sim_example_for_figure/parameter_set_posterior_mode__baseline_pass-grand_mean")
PANEL_C_OUTPUT_DIR = OUTPUT_DIR
PANEL_C_DISPLAY_WINDOW_REL_CUE_S = (-3.0, 1.0)
PANEL_C_INPUT_RUN_INDEX = 0

PANEL_C_INPUT_SHOW_INDIVIDUAL = False
PANEL_C_INPUT_SHOW_MEAN = False
PANEL_C_OUTPUT_SHOW_INDIVIDUAL = False # True
PANEL_C_OUTPUT_SHOW_MEAN = True

# Raw CST display is reconstructed from spike times and shown as unsmoothed pooled spike-count impulses.
PANEL_C_RAW_CST_SHOW_SELECTED = True
PANEL_C_RAW_CST_SHOW_INDIVIDUAL = False
PANEL_C_RAW_CST_SHOW_MEAN = False

PANEL_C_FEATURE_CONFIG = {
    "enabled": True,
    "fitting_window": (-3.0, 1.0),
    "analysis_window": (0.0, 0.5),
    "buffer_s": 0.05,
    "spline_smoothing": None,
    "spline_order": 4,
    "n_knots": None,
    "trend_pre_window": (-3.0, -1.0),
    "trend_post_window": (0.0, 1.0),
    "extrapolate": False,
}

PANEL_C_SAVE_FIGURE_FORMATS = ("png", "pdf")
PANEL_C_FEATURE_VALUES_CSV = PANEL_C_OUTPUT_DIR / "panel_C_sim_pipeline_feature_values.csv"


# Optional manual y-axis limits for Panel C subplots.
# Use a dict mapping subplot keys to (ymin, ymax). Missing keys or values set to None use automatic limits.
# Accepted keys:
#   Input figure: "input_tonic_context", "input_lf", "input_alpha", "input_beta", "input_total_pool"
#   Spike/CST figure: "spike_raster", "raw_cst_spike_count"
#   Processed figure: "cst_lf", "cst_alpha_mod", "cst_beta_mod", "sync_trace"
# Example: {"input_total_pool": (35000, 47000), "cst_alpha_mod": (-1.0, 2.0)}
PANEL_C_YLIM_OVERRIDES = {
    "input_tonic_context": (30_000, 48_000),
    "input_total_pool": (0, 60_000),
    #
    # "cst_lf": (12, 16),
    # "cst_alpha_mod": (-1.0, 3.0),
    # "cst_beta_mod": (-1.0, 3.0),
    # "sync_trace": (0.0, 1.0),
    #
    "raw_cst_spike_count": (0, 5),
}

# Panel C input-only y-scale matching. This keeps each subplot centered on its own trace,
# but uses the same y span across selected input subplots. Absolute PANEL_C_YLIM_OVERRIDES win.
PANEL_C_INPUT_SHARED_YSCALE = True
PANEL_C_INPUT_SHARED_YSCALE_KEYS = [
    "input_tonic_context",
    "input_lf",
    "input_alpha",
    "input_beta",
    "input_total_pool",
]
PANEL_C_INPUT_SHARED_YSCALE_PADDING = 1.10
# Accepted: "data_midpoint", "mean", "median".
PANEL_C_INPUT_SHARED_YSCALE_CENTER = "data_midpoint"

PANEL_C_STYLE = {
    "figure_dpi": GLOBAL_FIGURE_DPI,
    "input_figure_size": _figure_size("panel_c_input", (7.0, 8.0)),
    "spike_cst_figure_size": _figure_size("panel_c_spike_cst", (7.0, 4.8)),
    "processed_figure_size": _figure_size("panel_c_processed", (7.2, 7.2)),
    "title_fontsize": GLOBAL_TITLE_FONTSIZE,
    "axis_label_fontsize": GLOBAL_AXIS_LABEL_FONTSIZE,
    "xlabel_fontsize": GLOBAL_AXIS_LABEL_FONTSIZE,
    "ylabel_fontsize": GLOBAL_AXIS_LABEL_FONTSIZE,
    "tick_labelsize": GLOBAL_TICK_LABELSIZE,
    "legend_fontsize": GLOBAL_LEGEND_FONTSIZE,
    "suptitle_fontsize": GLOBAL_TITLE_FONTSIZE,
    "suptitle_y": None,
    "title_loc": "left",
    "legend_loc": "upper left",
    "legend_frameon": False,
    "tight_layout": False, # True,
    "subplots_adjust": {},
    "selected_lw": 1.0, # 1.6,
    "mean_lw": 1.0, # 2.0,
    "individual_lw": 0.5, # 0.7,
    "individual_alpha": 0.13,
    "envelope_lw": 1.0, # 0.9,
    "envelope_alpha": 0.75,
    "component_overlay_lw": 1.0, # 1.1,
    "component_overlay_alpha": 0.95,
    "component_trend_ls": "--",
    "component_burst_ls": ":",
    "fit_lw": 1.0, # 1.3,
    "residual_alpha": 0.24,
    "analysis_window_alpha": 0.08,
    "burst_epoch_alpha": 0.10,
    "grid_alpha": 0.22,
    "event_lw": 1.0, # 0.9,
    "event_alpha": 0.9,
    "go_event_alpha": 0.95,
    "burst_marker_alpha": 0.9,
    "marker_alpha": 0.9,
    "raster_marker_size": 3.0,
    "raster_marker_width": 0.5, # 0.8,
    "raster_tick_half_height": 0.36,
    "raster_alpha": 0.82,
    "raw_cst_impulse_lw": 1.0, # 0.75,
    "raw_cst_mean_lw": 1.0, # 1.2,
    "raw_cst_mean_alpha": 0.65,
    "raw_cst_selected_alpha": 0.92,
    "colors": {
        "baseline": "#6e6e6e",
        "cue_modulation": "#111111",
        "trend": "#4d4d4d",
        "lf": ROW_COLORS.get("cst_lf", "#ff3300"),
        "alpha": ROW_COLORS.get("cst_alpha_mod", "#ffa600"),
        "beta": ROW_COLORS.get("cst_beta_mod", "#9d7aff"),
        "sync": ROW_COLORS.get("sync_trace", "#a51e9e"),
        "common": "#111111",
        "fit": "#222222",
        "event": "#444444",
        "burst": "#3d7cff",
        "residual": "#2f80ed",
        "raster": "#222222",
        "raw_cst": "#555555",
    },
}


In [ ]:
def _panel_c_sorted_run_dirs(parameter_set_dir):
    parameter_set_dir = Path(parameter_set_dir)
    run_dirs = [p for p in parameter_set_dir.glob("simulation_output_*") if p.is_dir()]
    def _key(path):
        suffix = path.name.replace("simulation_output_", "")
        return int(suffix) if suffix.isdigit() else path.name
    return sorted(run_dirs, key=_key)


def _panel_c_load_runs(parameter_set_dir):
    run_dirs = _panel_c_sorted_run_dirs(parameter_set_dir)
    runs = []
    for run_dir in run_dirs:
        try:
            run = load_parameter_set_run(run_dir)
        except Exception as exc:
            warnings.warn(f"Skipping {run_dir}: {exc}")
            continue
        if run is not None:
            runs.append(run)
    if not runs:
        raise FileNotFoundError(f"No readable simulation runs found in {parameter_set_dir}")
    return runs


def _panel_c_h5_path_for_run(run):
    return Path(run["run_dir"]) / "simulation_output.h5"


def _panel_c_read_dataset(run, group_name, dataset_name):
    h5_path = _panel_c_h5_path_for_run(run)
    if not h5_path.exists():
        return None
    with h5py.File(h5_path, "r") as f:
        grp = f.get(group_name)
        if grp is None or dataset_name not in grp:
            return None
        return np.asarray(grp[dataset_name], dtype=float)


def _panel_c_read_first_available(run, candidates):
    for group_name, dataset_name in candidates:
        arr = _panel_c_read_dataset(run, group_name, dataset_name)
        if arr is not None:
            return arr
    return None


def _panel_c_go_cue_s(run):
    event_times = run.get("event_times", {})
    if "go_nogo_cue_s" in event_times:
        return float(event_times["go_nogo_cue_s"])
    params = run.get("params", {})
    return float(params.get("edges_ignore_duration", 0.0)) + float(params.get("get_ready_cue_time_s", 0.0)) + float(params.get("go_nogo_cue_delay_s", 0.0))


def _panel_c_time_rel(run):
    return np.asarray(run["t_s"], dtype=float) - _panel_c_go_cue_s(run)


def _panel_c_window_mask(time_rel):
    lo, hi = map(float, PANEL_C_DISPLAY_WINDOW_REL_CUE_S)
    return (time_rel >= lo) & (time_rel <= hi)


def _panel_c_interp_to_ref(source_time, source_trace, ref_time):
    source_time = np.asarray(source_time, dtype=float)
    source_trace = np.asarray(source_trace, dtype=float)
    ref_time = np.asarray(ref_time, dtype=float)
    if source_time.shape == ref_time.shape and np.allclose(source_time, ref_time, equal_nan=True):
        return source_trace
    return np.interp(ref_time, source_time, source_trace, left=np.nan, right=np.nan)


def _panel_c_stack_run_trace(runs, trace_getter, time_getter=None):
    time_getter = _panel_c_time_rel if time_getter is None else time_getter
    ref_time = None
    traces = []
    used = []
    for run in runs:
        trace = trace_getter(run)
        if trace is None:
            continue
        time = np.asarray(time_getter(run), dtype=float)
        trace = np.asarray(trace, dtype=float)
        if time.size != trace.size:
            warnings.warn(f"Skipping trace with mismatched time/trace sizes in {run['run_dir']}")
            continue
        if ref_time is None:
            ref_time = time
        traces.append(_panel_c_interp_to_ref(time, trace, ref_time))
        used.append(run)
    if ref_time is None or not traces:
        return None, None, []
    return ref_time, np.stack(traces, axis=0), used


def _panel_c_save_figure(fig, stem):
    PANEL_C_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    formats = PANEL_C_SAVE_FIGURE_FORMATS
    if isinstance(formats, str):
        formats = (formats,)
    paths = []
    for ext in formats:
        out_path = PANEL_C_OUTPUT_DIR / f"{stem}.{ext}"
        fig.savefig(out_path, bbox_inches=GLOBAL_SAVE_BBOX, pad_inches=GLOBAL_SAVE_PAD_INCHES, dpi=fig.dpi)
        paths.append(out_path)
    return paths


def _panel_c_draw_markers(ax, run, *, include_feature_windows=False):
    colors = PANEL_C_STYLE["colors"]
    event_times = run.get("event_times", {})
    go = _panel_c_go_cue_s(run)
    get_ready = float(event_times.get("get_ready_cue_s", go - 1.0)) - go
    burst_center = float(event_times.get("burst_center_s", go + 0.25)) - go
    burst_start = float(event_times.get("burst_start_s", burst_center + go - 0.2)) - go
    burst_end = burst_center + (burst_center - burst_start)
    ax.axvline(get_ready, color=colors["event"], lw=PANEL_C_STYLE["event_lw"], ls=":", alpha=0.9)
    ax.axvline(0.0, color=colors["event"], lw=PANEL_C_STYLE["event_lw"], ls="--", alpha=0.95)
    ax.axvline(burst_center, color=colors["burst"], lw=PANEL_C_STYLE["event_lw"], ls="--", alpha=0.9)
    ax.axvspan(burst_start, burst_end, color=colors["burst"], alpha=PANEL_C_STYLE["burst_epoch_alpha"], lw=0)
    if include_feature_windows:
        analysis_window = PANEL_C_FEATURE_CONFIG["analysis_window"]
        trend_pre = PANEL_C_FEATURE_CONFIG["trend_pre_window"]
        trend_post = PANEL_C_FEATURE_CONFIG["trend_post_window"]
        ax.axvspan(*analysis_window, color=colors["residual"], alpha=PANEL_C_STYLE["analysis_window_alpha"], lw=0)
        ax.axvspan(*trend_pre, color="0.5", alpha=0.045, lw=0)
        ax.axvspan(*trend_post, color="0.5", alpha=0.045, lw=0)


def _panel_c_plot_trace_stack(ax, time, stack, *, color, ylabel, title, show_individual, show_mean, selected_index=None):
    mask = _panel_c_window_mask(time)
    if show_individual:
        for trace in stack:
            ax.plot(time[mask], trace[mask], color=color, lw=PANEL_C_STYLE["individual_lw"], alpha=PANEL_C_STYLE["individual_alpha"])
    if selected_index is not None and 0 <= int(selected_index) < stack.shape[0]:
        ax.plot(time[mask], stack[int(selected_index), mask], color=color, lw=PANEL_C_STYLE["selected_lw"], alpha=0.95)
    if show_mean:
        ax.plot(time[mask], np.nanmean(stack[:, mask], axis=0), color=color, lw=PANEL_C_STYLE["mean_lw"], alpha=1.0)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=10)
    ax.grid(alpha=PANEL_C_STYLE["grid_alpha"])


def _panel_c_load_spike_trains(run):
    h5_path = _panel_c_h5_path_for_run(run)
    spike_trains = []
    with h5py.File(h5_path, "r") as f:
        mn_grp = f.get("spike_trains/MN")
        if mn_grp is None:
            raise KeyError(f"Missing spike_trains/MN in {h5_path}")
        keys = sorted(mn_grp.keys(), key=lambda k: int(k.split("_")[-1]) if k.split("_")[-1].isdigit() else k)
        for key in keys:
            spike_trains.append(np.asarray(mn_grp[key], dtype=float))
    return spike_trains


def _panel_c_apply_ylim(ax, key):
    limits = PANEL_C_YLIM_OVERRIDES.get(str(key)) if isinstance(PANEL_C_YLIM_OVERRIDES, dict) else None
    if limits is None:
        return
    if len(limits) != 2:
        raise ValueError(f"PANEL_C_YLIM_OVERRIDES[{key!r}] must be a (ymin, ymax) tuple or None")
    lo, hi = limits
    if lo is None or hi is None:
        return
    ax.set_ylim(float(lo), float(hi))


def _panel_c_cst_spike_count_trace(run):
    """Reconstruct unsmoothed pooled spike counts per saved CST time bin from MN spike times."""
    time_abs = np.asarray(run.get("t_s", []), dtype=float)
    if time_abs.size < 2:
        time_abs = _panel_c_read_dataset(run, "cst_diagnostics", "time_s")
        if time_abs is None or np.asarray(time_abs).size < 2:
            return None
        time_abs = np.asarray(time_abs, dtype=float)
    dt = float(np.nanmedian(np.diff(time_abs)))
    if not np.isfinite(dt) or dt <= 0:
        return None
    edges = np.concatenate([time_abs, [time_abs[-1] + dt]])
    counts = np.zeros(time_abs.size, dtype=float)
    for spikes in _panel_c_load_spike_trains(run):
        if np.asarray(spikes).size == 0:
            continue
        hist, _ = np.histogram(np.asarray(spikes, dtype=float), bins=edges)
        counts += hist.astype(float)
    return counts


In [ ]:
def plot_panel_c_input_components(runs):
    selected_run = runs[int(np.clip(PANEL_C_INPUT_RUN_INDEX, 0, len(runs) - 1))]
    colors = PANEL_C_STYLE["colors"]
    fig, axes = plt.subplots(5, 1, figsize=PANEL_C_STYLE["input_figure_size"], sharex=True, dpi=PANEL_C_STYLE["figure_dpi"])

    baseline_trace = _panel_c_read_first_available(selected_run, [("input_components", "baseline_tonic_trace_nA"), ("driving_inputs", "final_baseline_ts_nA")])
    if baseline_trace is None or baseline_trace.size == 1:
        baseline_value = float(selected_run["params"].get("excitatory_input_baseline", 0.0))
        baseline_trace = np.full_like(selected_run["common_signal"], baseline_value, dtype=float)
    # LF burst/trend is additive to the net low-frequency drive, not an envelope on an ongoing carrier.
    # For this schematic panel, show that cue modulation as a clean carving/addition to tonic baseline.
    cue_mod = _panel_c_read_first_available(selected_run, [("input_components", "lf_burst_kernel_nA"), ("driving_inputs", "lf_burst_kernel_nA")])
    cue_trend = _panel_c_read_first_available(selected_run, [("input_components", "lf_trend_kernel_nA"), ("driving_inputs", "lf_trend_kernel_nA")])
    if cue_mod is None:
        cue_mod = np.zeros_like(baseline_trace)
    if cue_trend is None:
        cue_trend = np.zeros_like(baseline_trace)
    tonic_plus_cue = baseline_trace + cue_mod + cue_trend

    lf_base = _panel_c_read_first_available(selected_run, [("input_components", "lf_final")])
    lf_final = lf_base
    if lf_final is None:
        lf_final = _panel_c_read_first_available(selected_run, [("input_components", "lf_final_with_burst"), ("input_components", "lf_final_with_trend")])
    alpha_final = _panel_c_read_first_available(selected_run, [("input_components", "alpha_final_with_burst"), ("input_components", "alpha_final_with_trend"), ("input_components", "alpha_final")])
    beta_final = _panel_c_read_first_available(selected_run, [("input_components", "beta_final_with_burst"), ("input_components", "beta_final_with_trend"), ("input_components", "beta_final")])
    alpha_env = _panel_c_read_first_available(selected_run, [("input_components", "alpha_effective_envelope_nA"), ("driving_inputs", "alpha_effective_envelope_nA")])
    beta_env = _panel_c_read_first_available(selected_run, [("input_components", "beta_effective_envelope_nA"), ("driving_inputs", "beta_effective_envelope_nA")])
    alpha_burst = _panel_c_read_first_available(selected_run, [("input_components", "alpha_burst_kernel_nA"), ("driving_inputs", "alpha_burst_kernel_nA")])
    alpha_trend = _panel_c_read_first_available(selected_run, [("input_components", "alpha_trend_kernel_nA"), ("driving_inputs", "alpha_trend_kernel_nA")])
    beta_burst = _panel_c_read_first_available(selected_run, [("input_components", "beta_burst_kernel_nA"), ("driving_inputs", "beta_burst_kernel_nA")])
    beta_trend = _panel_c_read_first_available(selected_run, [("input_components", "beta_trend_kernel_nA"), ("driving_inputs", "beta_trend_kernel_nA")])
    total_pool = _panel_c_read_first_available(selected_run, [("input_components", "total_pool_input_with_step_trace_nA"), ("input_components", "common_input_with_step")])
    if total_pool is None:
        total_pool = baseline_trace + selected_run["common_signal"]

    time_rel = _panel_c_time_rel(selected_run)
    mask = _panel_c_window_mask(time_rel)

    def _zero_kernel(kernel, reference):
        if kernel is None or reference is None:
            return np.zeros_like(reference, dtype=float) if reference is not None else None
        arr = np.asarray(kernel, dtype=float)
        return arr if arr.shape == np.asarray(reference).shape else np.resize(arr, np.asarray(reference).shape)

    def _envelope_component_lines(envelope, burst_kernel, trend_kernel):
        if envelope is None:
            return None, None
        envelope = np.asarray(envelope, dtype=float)
        burst_kernel = _zero_kernel(burst_kernel, envelope)
        trend_kernel = _zero_kernel(trend_kernel, envelope)
        base_envelope = np.maximum(envelope - burst_kernel - trend_kernel, 0.0)
        trend_line = np.maximum(base_envelope + trend_kernel, 0.0)
        burst_line = np.maximum(base_envelope + burst_kernel, 0.0)
        return trend_line, burst_line

    alpha_trend_line, alpha_burst_line = _envelope_component_lines(alpha_env, alpha_burst, alpha_trend)
    beta_trend_line, beta_burst_line = _envelope_component_lines(beta_env, beta_burst, beta_trend)

    input_y_values = {}

    def _record_input_y(key, *arrays):
        vals = []
        for arr in arrays:
            if arr is None:
                continue
            arr = np.asarray(arr, dtype=float)
            if arr.shape == mask.shape:
                arr = arr[mask]
            finite = arr[np.isfinite(arr)]
            if finite.size:
                vals.append(finite)
        if vals:
            input_y_values[str(key)] = np.concatenate(vals)

    def _center_for_values(values):
        center_mode = str(PANEL_C_INPUT_SHARED_YSCALE_CENTER).strip().lower()
        if center_mode == "mean":
            return float(np.nanmean(values))
        if center_mode == "median":
            return float(np.nanmedian(values))
        return 0.5 * (float(np.nanmin(values)) + float(np.nanmax(values)))

    def _apply_panel_c_input_shared_yscale(axes_by_key):
        if not bool(PANEL_C_INPUT_SHARED_YSCALE):
            return
        keys = [str(key) for key in PANEL_C_INPUT_SHARED_YSCALE_KEYS if str(key) in axes_by_key]
        spans = []
        for key in keys:
            if PANEL_C_YLIM_OVERRIDES.get(key) is not None:
                continue
            values = input_y_values.get(key)
            if values is None or values.size == 0:
                continue
            span = float(np.nanmax(values) - np.nanmin(values))
            if np.isfinite(span):
                spans.append(max(span, 1e-12))
        if not spans:
            return
        shared_span = max(spans) * float(PANEL_C_INPUT_SHARED_YSCALE_PADDING)
        for key in keys:
            if PANEL_C_YLIM_OVERRIDES.get(key) is not None:
                continue
            values = input_y_values.get(key)
            if values is None or values.size == 0:
                continue
            center = _center_for_values(values)
            axes_by_key[key].set_ylim(center - 0.5 * shared_span, center + 0.5 * shared_span)

    baseline_y = baseline_trace[mask]
    cue_y = tonic_plus_cue[mask]
    y_floor = float(np.nanmin([np.nanmin(baseline_y), np.nanmin(cue_y)]))
    y_span = max(float(np.nanmax([np.nanmax(baseline_y), np.nanmax(cue_y)]) - y_floor), 1.0)
    y_floor = y_floor - 0.08 * y_span
    axes[0].fill_between(time_rel[mask], y_floor, baseline_y, color=colors["baseline"], alpha=0.16, linewidth=0, label="Tonic baseline")
    axes[0].plot(time_rel[mask], baseline_y, color=colors["baseline"], lw=0.9, ls=":")
    axes[0].fill_between(time_rel[mask], baseline_y, cue_y, color=colors["cue_modulation"], alpha=0.24, linewidth=0, label="Burst/trend modulation")
    axes[0].plot(time_rel[mask], cue_y, color=colors["cue_modulation"], lw=PANEL_C_STYLE["selected_lw"], label="Baseline + burst/trend")
    axes[0].plot(time_rel[mask], (baseline_trace + cue_trend)[mask], color=colors.get("trend", "0.25"), lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_trend_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="LF trend")
    axes[0].plot(time_rel[mask], (baseline_trace + cue_mod)[mask], color=colors["burst"], lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_burst_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="LF burst Gaussian")
    _record_input_y("input_tonic_context", baseline_trace, tonic_plus_cue, baseline_trace + cue_trend, baseline_trace + cue_mod)
    axes[0].set_ylabel("Baseline\ninput (nA)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[0].set_title("Tonic baseline with raw burst/trend modulation", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[0].legend(frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])

    if lf_base is not None:
        axes[1].plot(time_rel[mask], lf_final[mask], color=colors["lf"], lw=PANEL_C_STYLE["selected_lw"], label="LF base fluctuations")
    _record_input_y("input_lf", lf_final)
    axes[1].set_ylabel("LF\n(nA)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[1].set_title("Low-frequency input fluctuations without cue modulation", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[1].legend(frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])

    axes[2].plot(time_rel[mask], alpha_final[mask], color=colors["alpha"], lw=PANEL_C_STYLE["selected_lw"], label="Alpha input")
    if alpha_env is not None:
        axes[2].plot(time_rel[mask], alpha_env[mask], color="0.15", lw=PANEL_C_STYLE["envelope_lw"], ls="--", alpha=PANEL_C_STYLE["envelope_alpha"], label="Envelope")
        axes[2].plot(time_rel[mask], -alpha_env[mask], color="0.15", lw=PANEL_C_STYLE["envelope_lw"], ls="--", alpha=PANEL_C_STYLE["envelope_alpha"])
    if alpha_trend_line is not None:
        axes[2].plot(time_rel[mask], alpha_trend_line[mask], color=colors.get("trend", "0.25"), lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_trend_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="Trend envelope")
    if alpha_burst_line is not None:
        axes[2].plot(time_rel[mask], alpha_burst_line[mask], color=colors["burst"], lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_burst_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="Burst Gaussian envelope")
    _record_input_y("input_alpha", alpha_final, alpha_env, -alpha_env if alpha_env is not None else None, alpha_trend_line, alpha_burst_line)
    axes[2].set_ylabel("Alpha\n(nA)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[2].set_title("Alpha input with effective envelope", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[2].legend(frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])

    axes[3].plot(time_rel[mask], beta_final[mask], color=colors["beta"], lw=PANEL_C_STYLE["selected_lw"], label="Beta input")
    if beta_env is not None:
        axes[3].plot(time_rel[mask], beta_env[mask], color="0.15", lw=PANEL_C_STYLE["envelope_lw"], ls="--", alpha=PANEL_C_STYLE["envelope_alpha"], label="Envelope")
        axes[3].plot(time_rel[mask], -beta_env[mask], color="0.15", lw=PANEL_C_STYLE["envelope_lw"], ls="--", alpha=PANEL_C_STYLE["envelope_alpha"])
    if beta_trend_line is not None:
        axes[3].plot(time_rel[mask], beta_trend_line[mask], color=colors.get("trend", "0.25"), lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_trend_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="Trend envelope")
    if beta_burst_line is not None:
        axes[3].plot(time_rel[mask], beta_burst_line[mask], color=colors["burst"], lw=PANEL_C_STYLE["component_overlay_lw"], ls=PANEL_C_STYLE["component_burst_ls"], alpha=PANEL_C_STYLE["component_overlay_alpha"], label="Burst Gaussian envelope")
    _record_input_y("input_beta", beta_final, beta_env, -beta_env if beta_env is not None else None, beta_trend_line, beta_burst_line)
    axes[3].set_ylabel("Beta\n(nA)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[3].set_title("Beta input with effective envelope", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[3].legend(frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])

    axes[4].plot(time_rel[mask], total_pool[mask], color=colors["common"], lw=PANEL_C_STYLE["selected_lw"])
    _record_input_y("input_total_pool", total_pool)
    axes[4].set_ylabel("Total pool\ninput (nA)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[4].set_title("Delivered pool input", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[4].set_xlabel("Time relative to go/no-go cue (s)", fontsize=PANEL_C_STYLE["xlabel_fontsize"])

    for ax, key in zip(axes, ["input_tonic_context", "input_lf", "input_alpha", "input_beta", "input_total_pool"]):
        _panel_c_draw_markers(ax, selected_run)
        ax.set_xlim(*PANEL_C_DISPLAY_WINDOW_REL_CUE_S)
        _panel_c_apply_ylim(ax, key)
        ax.tick_params(axis="both", labelsize=PANEL_C_STYLE["tick_labelsize"])
        ax.grid(alpha=PANEL_C_STYLE["grid_alpha"])
    _apply_panel_c_input_shared_yscale(dict(zip(["input_tonic_context", "input_lf", "input_alpha", "input_beta", "input_total_pool"], axes)))
    suptitle_kwargs = {"fontsize": PANEL_C_STYLE["suptitle_fontsize"]}
    if PANEL_C_STYLE["suptitle_y"] is not None:
        suptitle_kwargs["y"] = PANEL_C_STYLE["suptitle_y"]
    fig.suptitle(f"Panel C input components | {Path(PANEL_C_EXAMPLE_PARAMETER_SET_DIR).name} | run {PANEL_C_INPUT_RUN_INDEX}", **suptitle_kwargs)
    if PANEL_C_STYLE["tight_layout"]:
        fig.tight_layout()
    if PANEL_C_STYLE["subplots_adjust"]:
        fig.subplots_adjust(**PANEL_C_STYLE["subplots_adjust"])
    return fig


def plot_panel_c_spikes_and_raw_cst(runs):
    selected_run = runs[int(np.clip(PANEL_C_INPUT_RUN_INDEX, 0, len(runs) - 1))]
    colors = PANEL_C_STYLE["colors"]
    fig, axes = plt.subplots(2, 1, figsize=PANEL_C_STYLE["spike_cst_figure_size"], sharex=True, dpi=PANEL_C_STYLE["figure_dpi"], height_ratios=_subplot_control("panel_c_spike_cst_height_ratios", (1.25, 1.0)))

    go = _panel_c_go_cue_s(selected_run)
    spike_trains = _panel_c_load_spike_trains(selected_run)
    lo, hi = map(float, PANEL_C_DISPLAY_WINDOW_REL_CUE_S)
    for unit_i, spikes in enumerate(spike_trains):
        rel = np.asarray(spikes, dtype=float) - go
        rel = rel[(rel >= lo) & (rel <= hi)]
        if rel.size:
            half_height = float(PANEL_C_STYLE.get("raster_tick_half_height", 0.36))
            axes[0].vlines(
                rel,
                unit_i - half_height,
                unit_i + half_height,
                color=colors["raster"],
                linewidth=PANEL_C_STYLE.get("raster_marker_width", 0.8),
                alpha=PANEL_C_STYLE["raster_alpha"],
            )
    axes[0].set_ylabel("Motor unit", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[0].set_title("Individual motor-unit spike trains", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])

    selected_counts = _panel_c_cst_spike_count_trace(selected_run)
    selected_time = _panel_c_time_rel(selected_run)
    if selected_counts is None:
        raise ValueError("No raw CST spike-count trace could be reconstructed from spike_trains/MN")
    selected_counts = np.asarray(selected_counts, dtype=float)
    selected_mask = _panel_c_window_mask(selected_time)

    if bool(PANEL_C_RAW_CST_SHOW_INDIVIDUAL) or bool(PANEL_C_RAW_CST_SHOW_MEAN):
        time, stack, used = _panel_c_stack_run_trace(runs, _panel_c_cst_spike_count_trace)
        if stack is not None:
            stack_mask = _panel_c_window_mask(time)
            if bool(PANEL_C_RAW_CST_SHOW_INDIVIDUAL):
                for trace in stack:
                    trace = np.asarray(trace, dtype=float)
                    nz = stack_mask & (trace > 0)
                    axes[1].vlines(
                        time[nz],
                        0.0,
                        trace[nz],
                        color=colors["raw_cst"],
                        linewidth=PANEL_C_STYLE.get("raw_cst_impulse_lw", 0.75),
                        alpha=PANEL_C_STYLE.get("individual_alpha", 0.13),
                    )
            if bool(PANEL_C_RAW_CST_SHOW_MEAN):
                mean_trace = np.nanmean(stack, axis=0)
                axes[1].plot(
                    time[stack_mask],
                    mean_trace[stack_mask],
                    color=colors["raw_cst"],
                    lw=PANEL_C_STYLE.get("raw_cst_mean_lw", 1.2),
                    alpha=PANEL_C_STYLE["raw_cst_mean_alpha"],
                    label="Mean across runs",
                )

    if bool(PANEL_C_RAW_CST_SHOW_SELECTED):
        nz = selected_mask & (selected_counts > 0)
        axes[1].vlines(
            selected_time[nz],
            0.0,
            selected_counts[nz],
            color=colors["raw_cst"],
            linewidth=PANEL_C_STYLE.get("raw_cst_impulse_lw", 0.75),
            alpha=PANEL_C_STYLE["raw_cst_selected_alpha"],
            label=f"Selected run {PANEL_C_INPUT_RUN_INDEX}",
            zorder=5,
        )
    axes[1].set_ylabel("Raw CST\n(spikes/bin)", fontsize=PANEL_C_STYLE["ylabel_fontsize"])
    axes[1].set_title("Raw cumulative spike train (pooled spike count)", loc=PANEL_C_STYLE["title_loc"], fontsize=PANEL_C_STYLE["title_fontsize"])
    axes[1].legend(frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])
    n_units = len(spike_trains)
    for ax, key in zip(axes, ["spike_raster", "raw_cst_spike_count"]):
        _panel_c_draw_markers(ax, selected_run)
        ax.set_xlim(*PANEL_C_DISPLAY_WINDOW_REL_CUE_S)
        if key == "spike_raster" and PANEL_C_YLIM_OVERRIDES.get(key) is None:
            ax.set_ylim(-0.5, max(float(n_units) - 0.5, 0.5))
        elif key == "raw_cst_spike_count" and PANEL_C_YLIM_OVERRIDES.get(key) is None:
            ax.set_ylim(0.0, max(float(n_units), 1.0))
        _panel_c_apply_ylim(ax, key)
        ax.tick_params(axis="both", labelsize=PANEL_C_STYLE["tick_labelsize"])
        ax.grid(alpha=PANEL_C_STYLE["grid_alpha"])
    axes[1].set_xlabel("Time relative to go/no-go cue (s)", fontsize=PANEL_C_STYLE["xlabel_fontsize"])
    suptitle_kwargs = {"fontsize": PANEL_C_STYLE["suptitle_fontsize"]}
    if PANEL_C_STYLE["suptitle_y"] is not None:
        suptitle_kwargs["y"] = PANEL_C_STYLE["suptitle_y"]
    fig.suptitle(f"Panel C spike output | {Path(PANEL_C_EXAMPLE_PARAMETER_SET_DIR).name}", **suptitle_kwargs)
    if PANEL_C_STYLE["tight_layout"]:
        fig.tight_layout()
    if PANEL_C_STYLE["subplots_adjust"]:
        fig.subplots_adjust(**PANEL_C_STYLE["subplots_adjust"])
    return fig


def _panel_c_trace_specs():
    return [
        {"key": "cst_lf", "title": "Low-pass CST", "ylabel": "CST LF\n(spikes/s/MU)", "color": PANEL_C_STYLE["colors"]["lf"], "unit": "spikes/s/MU", "getter": lambda run: run.get("cst_lf"), "time_getter": _panel_c_time_rel},
        {"key": "cst_alpha_mod", "title": "Alpha CST modulation", "ylabel": "Alpha\n(z)", "color": PANEL_C_STYLE["colors"]["alpha"], "unit": "z", "getter": lambda run: run.get("cst_alpha_mod"), "time_getter": _panel_c_time_rel},
        {"key": "cst_beta_mod", "title": "Beta CST modulation", "ylabel": "Beta\n(z)", "color": PANEL_C_STYLE["colors"]["beta"], "unit": "z", "getter": lambda run: run.get("cst_beta_mod"), "time_getter": _panel_c_time_rel},
        {"key": "sync_trace", "title": "Rate-controlled synchrony", "ylabel": "Synchrony\nindex", "color": PANEL_C_STYLE["colors"]["sync"], "unit": "a.u.", "getter": lambda run: run.get("sync_trace"), "time_getter": lambda run: np.asarray(run.get("sync_time_abs"), dtype=float) - _panel_c_go_cue_s(run) if run.get("sync_time_abs") is not None else None},
    ]


def plot_panel_c_processed_features(runs):
    selected_run = runs[int(np.clip(PANEL_C_INPUT_RUN_INDEX, 0, len(runs) - 1))]
    colors = PANEL_C_STYLE["colors"]
    specs = _panel_c_trace_specs()
    fig, axes = plt.subplots(len(specs), 1, figsize=PANEL_C_STYLE["processed_figure_size"], sharex=True, dpi=PANEL_C_STYLE["figure_dpi"])
    feature_rows = []

    for ax, spec in zip(np.ravel(axes), specs):
        time, stack, used = _panel_c_stack_run_trace(runs, spec["getter"], time_getter=spec["time_getter"])
        if stack is None:
            warnings.warn(f"Skipping Panel C processed row {spec['key']}: no traces available")
            ax.axis("off")
            continue
        _panel_c_plot_trace_stack(
            ax, time, stack,
            color=spec["color"],
            ylabel=spec["ylabel"],
            title=spec["title"],
            show_individual=PANEL_C_OUTPUT_SHOW_INDIVIDUAL,
            show_mean=PANEL_C_OUTPUT_SHOW_MEAN,
        )
        mean_trace = np.nanmean(stack, axis=0)
        try:
            result = fit_baseline_spline_and_extract_feature(
                time,
                mean_trace,
                fitting_window=PANEL_C_FEATURE_CONFIG["fitting_window"],
                analysis_window=PANEL_C_FEATURE_CONFIG["analysis_window"],
                buffer_s=PANEL_C_FEATURE_CONFIG["buffer_s"],
                spline_smoothing=PANEL_C_FEATURE_CONFIG["spline_smoothing"],
                spline_order=PANEL_C_FEATURE_CONFIG["spline_order"],
                n_knots=PANEL_C_FEATURE_CONFIG["n_knots"],
                trend_pre_window=PANEL_C_FEATURE_CONFIG["trend_pre_window"],
                trend_post_window=PANEL_C_FEATURE_CONFIG["trend_post_window"],
                extrapolate=PANEL_C_FEATURE_CONFIG["extrapolate"],
                return_debug_info=True,
            )
            fit_t = np.asarray(result.t_fit_window, dtype=float)
            fit_y = np.asarray(result.baseline_fit_window, dtype=float)
            fit_mask = _panel_c_window_mask(fit_t)
            ax.plot(fit_t[fit_mask], fit_y[fit_mask], color=colors["fit"], lw=PANEL_C_STYLE["fit_lw"], ls="--", label="Spline fit")
            analysis_mask = np.asarray(result.analysis_mask, dtype=bool)
            ax.fill_between(
                fit_t[analysis_mask],
                np.asarray(result.y_fit_window)[analysis_mask],
                fit_y[analysis_mask],
                color=colors["residual"],
                alpha=PANEL_C_STYLE["residual_alpha"],
                linewidth=0,
                label="Burst residual",
            )
            feature_rows.append({
                "trace_key": spec["key"],
                "n_runs_used": int(len(used)),
                "feature_mean_residual": float(result.feature_mean_residual),
                "feature_signed_area": float(result.feature_signed_area),
                "trend_post_minus_pre": float(result.trend_post_minus_pre),
                "fit_r2": float(result.fit_r2),
                "fit_r2_gain_over_linear": float(result.fit_r2_gain_over_linear),
                "feature_unit": spec["unit"],
            })
        except Exception as exc:
            warnings.warn(f"Could not fit Panel C feature row {spec['key']}: {exc}")
        _panel_c_draw_markers(ax, selected_run, include_feature_windows=True)
        ax.set_xlim(*PANEL_C_DISPLAY_WINDOW_REL_CUE_S)
        _panel_c_apply_ylim(ax, spec["key"])
        ax.tick_params(axis="both", labelsize=PANEL_C_STYLE["tick_labelsize"])
        ax.grid(alpha=PANEL_C_STYLE["grid_alpha"])
    axes[-1].set_xlabel("Time relative to go/no-go cue (s)", fontsize=PANEL_C_STYLE["xlabel_fontsize"])
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        axes[0].legend(handles, labels, frameon=PANEL_C_STYLE["legend_frameon"], fontsize=PANEL_C_STYLE["legend_fontsize"], loc=PANEL_C_STYLE["legend_loc"])
    suptitle_kwargs = {"fontsize": PANEL_C_STYLE["suptitle_fontsize"]}
    if PANEL_C_STYLE["suptitle_y"] is not None:
        suptitle_kwargs["y"] = PANEL_C_STYLE["suptitle_y"]
    fig.suptitle(f"Panel C processed features | mean over {len(runs)} realizations", **suptitle_kwargs)
    if PANEL_C_STYLE["tight_layout"]:
        fig.tight_layout()
    if PANEL_C_STYLE["subplots_adjust"]:
        fig.subplots_adjust(**PANEL_C_STYLE["subplots_adjust"])
    return fig, pd.DataFrame(feature_rows)


In [ ]:
panel_c_runs = []
panel_c_feature_values_df = pd.DataFrame()
panel_c_figure_paths = []

if GENERATE_PANEL_C_SIM_PIPELINE_EXAMPLE:
    panel_c_runs = _panel_c_load_runs(PANEL_C_EXAMPLE_PARAMETER_SET_DIR)
    if len(panel_c_runs) != 32:
        warnings.warn(f"Panel C example expected 32 runs, found {len(panel_c_runs)}")

    fig_panel_c_input = plot_panel_c_input_components(panel_c_runs)
    panel_c_figure_paths.extend(_panel_c_save_figure(fig_panel_c_input, "panel_C_sim_pipeline_input_components"))

    fig_panel_c_spikes = plot_panel_c_spikes_and_raw_cst(panel_c_runs)
    panel_c_figure_paths.extend(_panel_c_save_figure(fig_panel_c_spikes, "panel_C_sim_pipeline_spikes_raw_cst"))

    fig_panel_c_processed, panel_c_feature_values_df = plot_panel_c_processed_features(panel_c_runs)
    panel_c_figure_paths.extend(_panel_c_save_figure(fig_panel_c_processed, "panel_C_sim_pipeline_processed_features"))

    PANEL_C_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    panel_c_feature_values_df.to_csv(PANEL_C_FEATURE_VALUES_CSV, index=False)

    print("Panel C outputs:")
    for p in panel_c_figure_paths:
        print(f" - {p}")
    print(f" - {PANEL_C_FEATURE_VALUES_CSV}")
    plt.show()
else:
    print("Panel C generation skipped because GENERATE_PANEL_C_SIM_PIPELINE_EXAMPLE=False")

panel_c_feature_values_df


## Feature-match correlation KDE

Posterior-predictive feature-vector agreement with the experimental target, summarized by Pearson correlation.


In [ ]:
def load_feature_match_correlation_metrics(path=FEATURE_MATCH_CORRELATION_METRICS_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Feature-match correlation table not found: {path}")
    df = pd.read_pickle(path)
    required = {"condition_id", "r"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Feature-match table is missing required columns: {sorted(missing)}")
    return df.copy()


def plot_feature_match_correlation_kde(metrics_df):
    style = dict(FEATURE_MATCH_CORRELATION_STYLE)
    labels = [cond for cond in CONDITION_ORDER if cond in set(metrics_df["condition_id"].astype(str))]
    if not labels:
        raise ValueError("No requested CONDITION_ORDER entries are present in the feature-match correlation table")
    width_ratios = style.get("width_ratios")
    gridspec_kw = {"width_ratios": width_ratios} if width_ratios is not None else None
    fig, axes = plt.subplots(
        1,
        len(labels),
        figsize=style["figure_size"],
        dpi=style["figure_dpi"],
        sharey=True,
        gridspec_kw=gridspec_kw,
    )
    axes = np.atleast_1d(axes)
    xlim = tuple(style.get("xlim", (0.0, 1.0)))
    grid = np.linspace(float(xlim[0]), float(xlim[1]), int(style.get("kde_points", 300)))
    summary_rows = []

    for ax, cond in zip(axes, labels):
        cond_df = metrics_df.loc[metrics_df["condition_id"].astype(str) == str(cond)].copy()
        values = pd.to_numeric(cond_df["r"], errors="coerce").to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        color = PANEL_B_CONDITION_COLORS.get(cond, "#333333")
        if values.size >= 2 and np.nanstd(values) > 0:
            kde = gaussian_kde(values)
            bandwidth_scale = float(style.get("kde_bandwidth_scale", 1.0))
            if np.isfinite(bandwidth_scale) and bandwidth_scale > 0:
                kde.set_bandwidth(kde.factor * bandwidth_scale)
            plot_grid = grid
            if bool(style.get("kde_truncate_to_observed_range", False)):
                padding = float(style.get("kde_truncate_padding", 0.0))
                lo = max(float(xlim[0]), float(np.nanmin(values)) - padding)
                hi = min(float(xlim[1]), float(np.nanmax(values)) + padding)
                plot_grid = grid[(grid >= lo) & (grid <= hi)]
                if plot_grid.size < 2:
                    plot_grid = grid
            density = kde(plot_grid)
            ax.plot(plot_grid, density, color=color, lw=style["density_lw"], alpha=style["density_alpha"], clip_on=True)
            ax.fill_between(plot_grid, 0.0, density, color=color, alpha=style["fill_alpha"], linewidth=0, clip_on=True)
        elif values.size == 1:
            ax.axvline(values[0], color=color, lw=style["density_lw"], alpha=style["density_alpha"])
        mean_r = float(np.nanmean(values)) if values.size else np.nan
        median_r = float(np.nanmedian(values)) if values.size else np.nan
        sd_r = float(np.nanstd(values, ddof=1)) if values.size > 1 else np.nan
        min_r = float(np.nanmin(values)) if values.size else np.nan
        max_r = float(np.nanmax(values)) if values.size else np.nan
        if np.isfinite(mean_r):
            ax.axvline(mean_r, color="black", lw=style["mean_lw"], ls=style["mean_ls"], label=f"mean r={mean_r:.2f}")
        ax.set_xlim(*xlim)
        ax.set_title(str(cond), fontsize=style["title_fontsize"], color=color)
        ax.set_xlabel(style["xlabel_text"], fontsize=style["axis_label_fontsize"])
        ax.tick_params(labelsize=style["tick_labelsize"])
        ax.grid(alpha=style["grid_alpha"])
        ax.legend(frameon=style["legend_frameon"], fontsize=style["legend_fontsize"], loc=style["legend_loc"])
        summary_rows.append({
            "condition_id": cond,
            "n_parameter_sets": int(values.size),
            "mean_r": mean_r,
            "median_r": median_r,
            "sd_r": sd_r,
            "min_r": min_r,
            "max_r": max_r,
        })
    axes[0].set_ylabel(style["ylabel_text"], fontsize=style["axis_label_fontsize"])
    suptitle_kwargs = {"fontsize": style["suptitle_fontsize"]}
    if style["suptitle_y"] is not None:
        suptitle_kwargs["y"] = style["suptitle_y"]
    fig.suptitle(style["suptitle_text"], **suptitle_kwargs)
    if style["tight_layout"]:
        fig.tight_layout()
    if style["subplots_adjust"]:
        fig.subplots_adjust(**style["subplots_adjust"])
    return fig, pd.DataFrame(summary_rows)


feature_match_correlation_metrics_df = pd.DataFrame()
feature_match_correlation_summary_df = pd.DataFrame()
feature_match_correlation_figure_paths = []

if GENERATE_PANEL_FEATURE_MATCH_CORRELATION:
    feature_match_correlation_metrics_df = load_feature_match_correlation_metrics()
    fig_feature_match_corr, feature_match_correlation_summary_df = plot_feature_match_correlation_kde(feature_match_correlation_metrics_df)
    feature_match_correlation_figure_paths = _save_figure(fig_feature_match_corr, OUTPUT_DIR, FEATURE_MATCH_CORRELATION_FILENAME_STEM, SAVE_FIGURE_FORMATS)
    print("Feature-match correlation outputs:")
    for p in feature_match_correlation_figure_paths:
        print(f" - {p}")
    plt.show()
else:
    print("Feature-match correlation figure skipped because GENERATE_PANEL_FEATURE_MATCH_CORRELATION=False")

feature_match_correlation_summary_df


## Feature-parameter correlation heatmap

Pearson correlations between sampled input parameters and simulated output features in the SBI training set.


In [ ]:
def _resolve_configured_names(available, requested, *, label):
    available = list(available)
    if requested is None:
        return available
    selected = [str(name) for name in requested]
    missing = [name for name in selected if name not in available]
    if missing:
        raise KeyError(f"Requested {label} not found in SBI dataset: {missing}")
    return selected


def _resolve_substring_color(name, color_map, fallback="black"):
    if not color_map:
        return fallback
    name_str = str(name)
    if name_str in color_map:
        return str(color_map[name_str])
    lower = name_str.lower()
    for key, color in color_map.items():
        if str(key).lower() in lower:
            return str(color)
    return fallback


def _resolve_feature_parameter_colormap(cmap_name):
    name = str(cmap_name)
    if name.lower() == "prinsenvlag":
        try:
            from cmcrameri import cm as cmc
            return getattr(cmc, "prinsenvlag")
        except Exception as exc:
            warnings.warn(
                f"Could not load cmcrameri prinsenvlag colormap ({exc}); "
                "using a local orange-white-blue prinsenvlag-style fallback."
            )
            return LinearSegmentedColormap.from_list(
                "prinsenvlag_fallback",
                ["#1f4e99", "#6f98c9", "#f7f7f2", "#f2a057", "#c94d20"],
                N=256,
            )
    return plt.get_cmap(name)


def load_feature_parameter_correlation_dataset(path=FEATURE_PARAMETER_CORRELATION_BUNDLE_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"SBI trained inference bundle not found: {path}")
    with path.open("rb") as f:
        bundle = pickle.load(f)
    if isinstance(bundle, dict):
        dataset = bundle.get("dataset")
    elif isinstance(bundle, tuple) and len(bundle) >= 1:
        dataset = bundle[0]
    else:
        dataset = bundle
    required_attrs = ["sim_rows_used", "selected_parameters", "selected_features"]
    missing = [name for name in required_attrs if not hasattr(dataset, name)]
    if missing:
        raise TypeError(f"Loaded bundle does not expose expected dataset attributes: {missing}")
    return dataset


def _pearson_corr_pair(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 2:
        return np.nan, int(mask.sum())
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) <= 0 or np.nanstd(y) <= 0:
        return np.nan, int(mask.sum())
    return float(np.corrcoef(x, y)[0, 1]), int(mask.sum())


def build_feature_parameter_correlation_table(dataset):
    if str(FEATURE_PARAMETER_CORRELATION_METHOD).lower() != "pearson":
        raise ValueError("Only Pearson correlations are implemented for this paper panel")
    parameters = _resolve_configured_names(
        dataset.selected_parameters,
        FEATURE_PARAMETER_CORRELATION_SELECTED_PARAMETERS,
        label="parameters",
    )
    features = _resolve_configured_names(
        dataset.selected_features,
        FEATURE_PARAMETER_CORRELATION_SELECTED_FEATURES,
        label="features",
    )
    rows = []
    sim_df = dataset.sim_rows_used
    for feature in features:
        y = pd.to_numeric(sim_df[feature], errors="coerce").to_numpy(dtype=float)
        for parameter in parameters:
            x = pd.to_numeric(sim_df[parameter], errors="coerce").to_numpy(dtype=float)
            r, n = _pearson_corr_pair(x, y)
            rows.append({
                "feature_name": feature,
                "parameter_name": parameter,
                "pearson_r": r,
                "n_simulations": n,
            })
    return pd.DataFrame(rows), features, parameters


def plot_feature_parameter_correlation_heatmap(corr_df, features, parameters):
    style = dict(FEATURE_PARAMETER_CORRELATION_STYLE)
    orientation = str(FEATURE_PARAMETER_CORRELATION_ORIENTATION).lower()
    if orientation not in {"features_rows", "parameters_rows"}:
        raise ValueError('FEATURE_PARAMETER_CORRELATION_ORIENTATION must be "features_rows" or "parameters_rows"')

    matrix = corr_df.pivot(index="feature_name", columns="parameter_name", values="pearson_r").reindex(index=features, columns=parameters).to_numpy(dtype=float)
    if orientation == "features_rows":
        plot_matrix = matrix
        x_names = parameters
        y_names = features
        x_formatter = format_parameter_label
        y_formatter = format_feature_label
        x_colors = style.get("parameter_colors")
        y_colors = style.get("feature_colors")
    else:
        plot_matrix = matrix.T
        x_names = features
        y_names = parameters
        x_formatter = format_feature_label
        y_formatter = format_parameter_label
        x_colors = style.get("feature_colors")
        y_colors = style.get("parameter_colors")

    fig_size = style.get("figure_size")
    if fig_size is None:
        cell = float(style.get("cell_size", 0.58))
        fig_size = (max(5.0, cell * len(x_names) + 2.8), max(4.5, cell * len(y_names) + 2.4))
    fig, ax = plt.subplots(1, 1, figsize=tuple(fig_size), dpi=int(style.get("figure_dpi", 600)))
    cmap = _resolve_feature_parameter_colormap(style.get("cmap", "prinsenvlag"))
    if bool(style.get("draw_heatmap_as_vector_cells", True)):
        x_edges = np.arange(len(x_names) + 1, dtype=float) - 0.5
        y_edges = np.arange(len(y_names) + 1, dtype=float) - 0.5
        im = ax.pcolormesh(
            x_edges,
            y_edges,
            plot_matrix,
            cmap=cmap,
            vmin=float(style.get("vmin", -1.0)),
            vmax=float(style.get("vmax", 1.0)),
            shading="flat",
            edgecolors=style.get("heatmap_edgecolor", "white"),
            linewidth=float(style.get("heatmap_edge_lw", 0.7)),
            rasterized=False,
        )
        ax.set_xlim(-0.5, len(x_names) - 0.5)
        ax.set_ylim(len(y_names) - 0.5, -0.5)
        if bool(style.get("square_cells", True)):
            ax.set_aspect("equal", adjustable="box")
    else:
        im = ax.imshow(
            plot_matrix,
            cmap=cmap,
            vmin=float(style.get("vmin", -1.0)),
            vmax=float(style.get("vmax", 1.0)),
            aspect="equal" if bool(style.get("square_cells", True)) else "auto",
            interpolation="nearest",
            rasterized=False,
        )
    ax.set_xticks(np.arange(len(x_names)))
    ax.set_yticks(np.arange(len(y_names)))
    ax.set_xticklabels([x_formatter(name) for name in x_names], rotation=float(style.get("x_tick_rotation", 45.0)), ha="right", fontsize=float(style.get("tick_labelsize", 7.0)))
    ax.set_yticklabels([y_formatter(name) for name in y_names], rotation=float(style.get("y_tick_rotation", 0.0)), fontsize=float(style.get("tick_labelsize", 7.0)))
    for tick, name in zip(ax.get_xticklabels(), x_names):
        tick.set_color(_resolve_substring_color(name, x_colors, fallback="black"))
    for tick, name in zip(ax.get_yticklabels(), y_names):
        tick.set_color(_resolve_substring_color(name, y_colors, fallback="black"))
    ax.set_xticks(np.arange(-0.5, len(x_names), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(y_names), 1), minor=True)
    ax.grid(which="minor", color=style.get("grid_color", "white"), linewidth=float(style.get("grid_lw", 0.7)))
    ax.tick_params(axis="both", which="major", length=float(style.get("tick_length", 3.0)), labelsize=float(style.get("tick_labelsize", 7.0)))
    ax.tick_params(which="minor", bottom=False, left=False)
    if bool(style.get("show_cell_values", True)):
        for row in range(plot_matrix.shape[0]):
            for col in range(plot_matrix.shape[1]):
                value = plot_matrix[row, col]
                label = "NA" if not np.isfinite(value) else f"{value:.2f}"
                threshold = float(style.get("value_text_abs_threshold", 0.55))
                text_color = "white" if np.isfinite(value) and abs(value) >= threshold else "black"
                ax.text(col, row, label, ha="center", va="center", fontsize=float(style.get("text_fontsize", 7.0)), color=text_color)
    cbar = fig.colorbar(im, ax=ax, shrink=float(style.get("colorbar_shrink", 0.82)))
    cbar.set_label(style.get("colorbar_label", "Pearson r"), fontsize=float(style.get("colorbar_label_fontsize", GLOBAL_AXIS_LABEL_FONTSIZE)))
    cbar.ax.tick_params(labelsize=float(style.get("colorbar_tick_labelsize", GLOBAL_TICK_LABELSIZE)))
    ax.set_title(style.get("title_text", "Feature-parameter correlations across all SBI training simulations"), fontsize=float(style.get("title_fontsize", 11.0)))
    fig.subplots_adjust(
        left=float(style.get("subplot_left", 0.24)),
        right=float(style.get("subplot_right", 0.86)),
        bottom=float(style.get("subplot_bottom", 0.28)),
        top=float(style.get("subplot_top", 0.88)),
    )
    return fig


feature_parameter_correlation_dataset = None
feature_parameter_correlation_table_df = pd.DataFrame()
feature_parameter_correlation_figure_paths = []

if GENERATE_PANEL_FEATURE_PARAMETER_CORRELATION:
    feature_parameter_correlation_dataset = load_feature_parameter_correlation_dataset()
    feature_parameter_correlation_table_df, feature_parameter_selected_features, feature_parameter_selected_parameters = build_feature_parameter_correlation_table(feature_parameter_correlation_dataset)
    fig_feature_parameter_corr = plot_feature_parameter_correlation_heatmap(
        feature_parameter_correlation_table_df,
        feature_parameter_selected_features,
        feature_parameter_selected_parameters,
    )
    feature_parameter_correlation_figure_paths = _save_figure(fig_feature_parameter_corr, OUTPUT_DIR, FEATURE_PARAMETER_CORRELATION_FILENAME_STEM, SAVE_FIGURE_FORMATS)
    print("Feature-parameter correlation outputs:")
    for p in feature_parameter_correlation_figure_paths:
        print(f" - {p}")
    plt.show()
else:
    print("Feature-parameter correlation figure skipped because GENERATE_PANEL_FEATURE_PARAMETER_CORRELATION=False")

feature_parameter_correlation_table_df
